[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/wdittaya/AI-for-digital-health-course/blob/main/lab05_object_detection_yolo.ipynb)

# Lab 5: การประมวลผลวิดีโอและการตรวจจับวัตถุ (Object Detection) ด้วย YOLO

**รายวิชา:** 3099704 AI for Digital Health (2026) · **คาบเรียน:** วันพุธที่ 14 ตุลาคม 2026 · **เวลาโดยประมาณ:** 90 นาที

## วัตถุประสงค์การเรียนรู้

- ใช้โมเดล YOLO ที่ผ่านการฝึกล่วงหน้า (pretrained) ตรวจจับวัตถุในภาพและวิดีโอ
- ปรับละเอียด (fine-tune) โมเดล YOLO ด้วยชุดข้อมูล (dataset) ทางการแพทย์สาธารณะ เพื่อตรวจหาเนื้องอกสมองในภาพ MRI
- ประเมินผลการตรวจจับด้วย mAP ตรวจพินิจผลการทำนาย และวิเคราะห์ความคลาดเคลื่อนของโมเดล
- เลือกเกณฑ์ความเชื่อมั่น (confidence threshold) และอธิบายการแลกเปลี่ยนระหว่างการตรวจไม่พบรอยโรคกับการแจ้งเตือนผิด
- อธิบายความเชื่อมโยงระหว่างการตรวจจับวัตถุกับกระบวนการทางคลินิก เช่น การตรวจหาติ่งเนื้อ (polyp) ในวิดีโอการส่องกล้องลำไส้ใหญ่

> **⚡ ปฏิบัติการนี้ต้องใช้ GPU** ก่อนเริ่มให้ตั้งค่าใน Colab ดังนี้ `Runtime → Change runtime type → T4 GPU → Save` หาก Colab แจ้งว่าไม่มี GPU ให้ใช้งานในขณะนั้น ผู้เรียนยังสามารถทำปฏิบัติการบน CPU ได้ โดยขั้นตอนการฝึกโมเดลจะใช้จำนวนรอบน้อยลงและใช้เวลานานขึ้น

> **แนวทางการทำปฏิบัติการ** (งานเดี่ยว)
>
> - ✅ **ส่วนหลัก — สำหรับผู้เรียนทุกคน ไม่จำเป็นต้องมีพื้นฐานการเขียนโปรแกรม** ให้รันเซลล์ (cell) ตามลำดับจากบนลงล่าง (คลิกปุ่ม ▶ หรือกด **Shift+Enter**) แบบฝึกหัด ⭐ **Exercise** แต่ละข้อกำหนดให้เปลี่ยนค่าเพียงหนึ่งค่าหรือเติมโค้ดสั้น ๆ หนึ่งบรรทัด และมีเซลล์ **✅ เฉลย** อยู่ถัดไป ผู้เรียนควรทำแบบฝึกหัดด้วยตนเองก่อน หากไม่สามารถดำเนินการต่อได้ ให้รันเซลล์เฉลยแล้วทำขั้นตอนถัดไป จากนั้นตอบคำถาม ✍️ ในเซลล์ข้อความ (ดับเบิลคลิกที่เซลล์ *✍️ คำตอบ* เพื่อพิมพ์) คำตอบเหล่านี้คือชิ้นงานที่ต้องส่ง หากพบปัญหา สามารถสอบถามผู้สอนผ่านช่องแชตของ Zoom
> - 🏆 **โจทย์ท้าทายบนตารางอันดับ — ไม่บังคับ สำหรับผู้เรียนที่เขียนโปรแกรมได้** อยู่ท้ายโน้ตบุ๊ก (notebook) และไม่เป็นส่วนหนึ่งของชิ้นงานที่ต้องส่ง

ปฏิบัติการทั้งหมดรันได้บน [Google Colab](https://colab.research.google.com) รุ่นฟรี ส่วนหลักใช้เวลาประมาณ 60–90 นาที (การฝึกโมเดลใช้เวลาประมาณ 3 นาทีบน GPU T4)

In [ ]:
# @title ⚙️ การตั้งค่า — โปรดรันเซลล์นี้ก่อน (NAME และ JOIN_CODE ใช้เฉพาะโจทย์ท้าทายบนตารางอันดับ ซึ่งไม่บังคับ)
# ============================================================================
# 🎯 วัตถุประสงค์ของเซลล์นี้
#   เซลล์นี้เป็นการ "เตรียมความพร้อม" ก่อนเริ่มปฏิบัติการ โดยทำ 3 อย่าง ได้แก่
#   1) ติดตั้งเครื่องมือ (ไลบรารี) เพิ่มเติมที่จำเป็นต้องใช้ใน Google Colab
#   2) กำหนดค่าพื้นฐาน เช่น รหัสปฏิบัติการ ชื่อที่จะแสดง และรหัสเข้าร่วมตารางอันดับ
#   3) สร้าง "ฟังก์ชัน" (ชุดคำสั่งสำเร็จรูปที่เรียกใช้ซ้ำได้) 2 ตัว คือ
#        - lab_data(...)              : ดาวน์โหลดไฟล์ข้อมูลของรายวิชา
#        - submit_to_leaderboard(...) : ส่งผลการทำนายไปยังตารางอันดับ (ไม่บังคับ)
#   ผู้เรียนเพียงกดรันเซลล์นี้ 1 ครั้ง ไม่จำเป็นต้องแก้ไขโค้ด
#   (ยกเว้นต้องการเข้าร่วมตารางอันดับ ให้กรอก NAME และ JOIN_CODE ด้านล่าง)
# ============================================================================

# ติดตั้งไลบรารีเพิ่มเติมลงในเครื่อง Colab
#   - เครื่องหมาย ! ด้านหน้า หมายถึงสั่งงานระบบปฏิบัติการ (ไม่ใช่คำสั่งภาษา Python)
#   - pip install : คำสั่งติดตั้งไลบรารีของ Python
#   - -q (quiet)  : ให้แสดงข้อความระหว่างติดตั้งให้น้อยที่สุด
#   - gradio_client   : ไลบรารีสำหรับติดต่อกับเว็บแอปตารางอันดับ (Hugging Face Space)
#   - huggingface_hub : ไลบรารีสำหรับดาวน์โหลดไฟล์จากเว็บไซต์ Hugging Face
!pip -q install gradio_client huggingface_hub

# สร้างตัวแปร (ช่องเก็บค่า) HF_OWNER แล้วใส่ข้อความ "wdittaya" ไว้
#   เครื่องหมาย = หมายถึง "นำค่าทางขวา ไปเก็บไว้ในชื่อทางซ้าย"
#   ข้อความในภาษา Python ต้องอยู่ในเครื่องหมายคำพูด " "
HF_OWNER  = "wdittaya"     # บัญชี Hugging Face ที่ใช้โฮสต์ตารางอันดับของรายวิชา (ผู้สอนเป็นผู้กำหนด)
# รหัสของปฏิบัติการนี้ ใช้ระบุว่าผลที่ส่งเป็นของปฏิบัติการใด (ไม่ต้องแก้ไข)
LAB_ID    = "lab05"
# ชื่อที่จะแสดงบนตารางอันดับ: พิมพ์ชื่อไว้ระหว่างเครื่องหมาย " " เช่น NAME = "หมอเอ"
NAME      = ""            # ไม่บังคับ: ชื่อที่จะแสดงบนตารางอันดับแบบสาธารณะ (สามารถใช้นามแฝงได้)
# รหัสเข้าร่วมตารางอันดับ: พิมพ์รหัสที่ผู้สอนแจ้งไว้ระหว่างเครื่องหมาย " "
JOIN_CODE = ""            # ไม่บังคับ: รหัสเข้าร่วมซึ่งผู้สอนประกาศในการเรียนการสอนผ่าน Zoom

# สร้างชื่อเว็บแอปตารางอันดับ โดยนำ HF_OWNER มาต่อกับข้อความ
#   - f"..." (f-string) คือข้อความที่แทรกค่าตัวแปรได้ โดยเขียนชื่อตัวแปรไว้ใน { }
#   - ผลลัพธ์คือ "wdittaya/aidh-leaderboard"
SPACE_ID  = f"{HF_OWNER}/aidh-leaderboard"
# สร้างชื่อคลังข้อมูล (dataset) ของรายวิชาในลักษณะเดียวกัน ผลลัพธ์คือ "wdittaya/aidh-lab-data"
DATA_REPO = f"{HF_OWNER}/aidh-lab-data"

# นำเข้า (import) ฟังก์ชัน hf_hub_download จากไลบรารี huggingface_hub
#   ฟังก์ชันนี้ใช้ดาวน์โหลดไฟล์จากเว็บไซต์ Hugging Face มาเก็บไว้ในเครื่อง
from huggingface_hub import hf_hub_download

# ----------------------------------------------------------------------------
# สร้างฟังก์ชัน lab_data สำหรับดาวน์โหลดไฟล์ข้อมูลของรายวิชา
#   - def คือคำสั่งสร้างฟังก์ชันใหม่ ตามด้วยชื่อฟังก์ชันและพารามิเตอร์ในวงเล็บ
#   - path : พารามิเตอร์ (ค่าที่ผู้ใช้ส่งเข้ามา) คือชื่อไฟล์ที่ต้องการ เช่น "pima/test_ids.csv"
#   - บรรทัดที่ย่อหน้าเข้าไปด้านล่าง คือคำสั่งที่อยู่ภายในฟังก์ชัน
# ----------------------------------------------------------------------------
def lab_data(path):
    """ดาวน์โหลดไฟล์ข้อมูลทดสอบสาธารณะของรายวิชา 1 ไฟล์ (เช่น 'pima/test_ids.csv') และคืนค่าตำแหน่งไฟล์ในเครื่อง"""
    # เรียก hf_hub_download(...) เพื่อดาวน์โหลดไฟล์ แล้วส่งตำแหน่งไฟล์ในเครื่องกลับไป (return)
    #   - DATA_REPO           : ชื่อคลังข้อมูลบน Hugging Face ที่เก็บไฟล์
    #   - path                : ชื่อไฟล์ที่ต้องการดาวน์โหลดภายในคลังข้อมูลนั้น
    #   - repo_type="dataset" : ระบุว่าคลังนี้เป็นประเภท "ชุดข้อมูล" (ไม่ใช่โมเดล)
    return hf_hub_download(DATA_REPO, path, repo_type="dataset")

# ----------------------------------------------------------------------------
# สร้างฟังก์ชัน submit_to_leaderboard สำหรับส่งไฟล์ผลการทำนายไปยังตารางอันดับ
#   - path   : ชื่อไฟล์ผลการทำนายที่ต้องการส่ง
#   - lab=LAB_ID : รหัสปฏิบัติการ หากไม่ระบุ จะใช้ค่า LAB_ID ที่กำหนดไว้ด้านบนโดยอัตโนมัติ
# ----------------------------------------------------------------------------
def submit_to_leaderboard(path, lab=LAB_ID):
    """ส่งไฟล์ผลการทำนายไปยังตารางอันดับของรายวิชา และแสดงคะแนนพร้อมอันดับ"""
    # นำเข้าเครื่องมือ 2 ตัวจากไลบรารี gradio_client
    #   - Client      : ใช้เชื่อมต่อกับเว็บแอปตารางอันดับ
    #   - handle_file : ใช้เตรียมไฟล์ในเครื่องให้อยู่ในรูปแบบที่ส่งขึ้นเว็บแอปได้
    from gradio_client import Client, handle_file
    # assert คือการตรวจสอบเงื่อนไข หากเงื่อนไขเป็นเท็จ โปรแกรมจะหยุดและแสดงข้อความเตือน
    #   - NAME and JOIN_CODE : เงื่อนไขว่าต้องกรอกทั้ง NAME และ JOIN_CODE แล้ว (ไม่เป็นข้อความว่าง)
    #   - ข้อความหลังเครื่องหมายจุลภาค คือข้อความเตือนที่จะแสดงเมื่อยังไม่ได้กรอก
    assert NAME and JOIN_CODE, "โปรดระบุ NAME และ JOIN_CODE ในเซลล์การตั้งค่าก่อน (จำเป็นเฉพาะการส่งผลไปยังตารางอันดับ)"
    # เชื่อมต่อกับเว็บแอปตารางอันดับ แล้วเก็บการเชื่อมต่อไว้ในตัวแปร client
    #   - SPACE_ID      : ชื่อเว็บแอปตารางอันดับที่กำหนดไว้ด้านบน
    #   - verbose=False : ไม่ต้องแสดงข้อความรายละเอียดระหว่างเชื่อมต่อ
    client = Client(SPACE_ID, verbose=False)
    # ส่งข้อมูลไปยังเว็บแอปด้วย client.predict(...) แล้วแสดงผลลัพธ์ (คะแนนและอันดับ) ด้วย print(...)
    #   - lab=lab                 : รหัสปฏิบัติการ
    #   - name=NAME               : ชื่อที่จะแสดงบนตารางอันดับ
    #   - code=JOIN_CODE          : รหัสเข้าร่วม
    #   - file=handle_file(path)  : ไฟล์ผลการทำนายที่เตรียมพร้อมสำหรับส่ง
    #   - api_name="/submit"      : ชื่อช่องทางรับข้อมูลของเว็บแอป (ส่วน "ส่งผล")
    print(client.predict(lab=lab, name=NAME, code=JOIN_CODE, file=handle_file(path), api_name="/submit"))

# แสดงข้อความยืนยันว่าการตั้งค่าเสร็จแล้ว ด้วยฟังก์ชัน print(...) (พิมพ์ข้อความออกทางหน้าจอ)
#   - ข้อความเป็น f-string จึงแทรกค่า SPACE_ID, LAB_ID และ NAME ลงในข้อความได้
#   - NAME or '(ยังไม่ได้ระบุ ...)' : หาก NAME ว่าง จะแสดงข้อความในวงเล็บแทน
print(f"Leaderboard: https://huggingface.co/spaces/{SPACE_ID}  |  lab={LAB_ID}  name={NAME or '(ยังไม่ได้ระบุ — ตารางอันดับไม่บังคับ)'}")

## เครื่องมือ: Ultralytics YOLO

**การตรวจจับวัตถุ (object detection)** คือการให้โมเดลระบุ *ตำแหน่ง* ของสิ่งที่สนใจในภาพ โดยวาด **กรอบล้อมวัตถุ (bounding box)** พร้อม **ป้ายระบุคลาส (class label)** และ **ค่าความเชื่อมั่น (confidence score)** ระหว่าง 0 ถึง 1 ซึ่งแสดงระดับความมั่นใจของโมเดล **YOLO** ("You Only Look Once") เป็นตระกูลโมเดลตรวจจับวัตถุที่ประมวลผลได้รวดเร็ว แพ็กเกจ `ultralytics` มีโมเดล YOLO ที่ผ่านการฝึกล่วงหน้าและคำสั่งสำหรับฝึกโมเดลที่ใช้งานได้ในบรรทัดเดียว

เซลล์ถัดไปจะติดตั้งแพ็กเกจ (ประมาณ 1 นาที) โหลดโมเดลขนาดเล็ก (`yolo11n` โดย "n" หมายถึง nano ซึ่งเล็กและเร็วที่สุด) และตรวจสอบว่ามี GPU หรือไม่

In [ ]:
# ============================================================================
# 🎯 วัตถุประสงค์ของเซลล์นี้
#   เซลล์นี้เตรียม "เครื่องมือตรวจจับวัตถุ (object detection)" ให้พร้อมใช้งาน โดยทำ 4 อย่าง ได้แก่
#   1) ติดตั้งไลบรารี ultralytics ซึ่งเป็นชุดเครื่องมือสำหรับใช้งานโมเดล YOLO
#   2) นำเข้า (import) เครื่องมือที่จำเป็น ได้แก่ YOLO, torch และ matplotlib (ใช้วาดภาพ/กราฟ)
#   3) ตรวจสอบว่าเครื่อง Colab มี GPU (การ์ดประมวลผลกราฟิกที่ช่วยให้ฝึกโมเดลได้เร็วขึ้นมาก) หรือไม่
#   4) โหลดโมเดล YOLO ขนาดเล็กที่สุด (nano) ซึ่งผ่านการฝึกล่วงหน้า (pretrained) ด้วยภาพวัตถุทั่วไปแล้ว
#   ผลลัพธ์ที่ผู้เรียนจะเห็น: ข้อความบอกว่ามี GPU หรือไม่ และรายชื่อตัวอย่างคลาส (ชนิดวัตถุ) ที่โมเดลรู้จัก
#   ใช้เวลาประมาณ 1 นาที (ส่วนใหญ่เป็นเวลาติดตั้งไลบรารี)
# ============================================================================

# ติดตั้งไลบรารี ultralytics ลงในเครื่อง Colab
#   - เครื่องหมาย ! ด้านหน้า หมายถึงสั่งงานระบบปฏิบัติการ (ไม่ใช่คำสั่งภาษา Python)
#   - pip install : คำสั่งติดตั้งไลบรารีของ Python
#   - -q (quiet)  : ให้แสดงข้อความระหว่างติดตั้งให้น้อยที่สุด
#   - ultralytics : ไลบรารีของบริษัท Ultralytics ที่รวมโมเดล YOLO และคำสั่งฝึก/ทำนาย/ประเมินผลไว้ครบ
!pip -q install ultralytics
# นำเข้า (import) คลาส YOLO จากไลบรารี ultralytics
#   - from ... import ... : ดึงเฉพาะเครื่องมือที่ต้องการออกมาจากไลบรารี เพื่อเรียกใช้ด้วยชื่อสั้น ๆ ได้ทันที
#   - YOLO : เครื่องมือสำหรับสร้าง/โหลดโมเดล YOLO ("You Only Look Once") ซึ่งใช้ทั้งทำนาย ฝึก และประเมินผล
from ultralytics import YOLO
# นำเข้าไลบรารี 2 ตัวในบรรทัดเดียว (คั่นด้วยเครื่องหมายจุลภาค ,)
#   - torch : ไลบรารี PyTorch สำหรับการเรียนรู้เชิงลึก (deep learning) ซึ่งเป็นพื้นฐานของ YOLO
#             ในเซลล์นี้ใช้ตรวจสอบว่ามี GPU หรือไม่
#   - matplotlib.pyplot as plt : ไลบรารีวาดภาพและกราฟ โดย "as plt" คือการตั้งชื่อย่อเป็น plt
#             เพื่อให้เขียนสั้นลง เช่น plt.imshow(...) แทน matplotlib.pyplot.imshow(...)
import torch, matplotlib.pyplot as plt

# แสดงข้อความว่าเครื่องนี้มี GPU หรือไม่ ด้วย print(...) (คำสั่งนี้เขียนยาว 2 บรรทัด แต่เป็นคำสั่งเดียว)
#   - print สามารถรับหลายค่าคั่นด้วยจุลภาค แล้วพิมพ์ต่อกันโดยเว้นวรรคให้อัตโนมัติ
#   - ค่าที่ 1 "มี GPU หรือไม่:" : ข้อความนำหน้า
#   - ค่าที่ 2 torch.cuda.is_available() : ฟังก์ชันของ PyTorch ที่ตอบ True (มี GPU) หรือ False (ไม่มี)
#             (CUDA คือเทคโนโลยีของ NVIDIA สำหรับใช้ GPU ประมวลผล)
#   - ค่าที่ 3 เป็นนิพจน์เงื่อนไขแบบบรรทัดเดียว รูปแบบ  A if เงื่อนไข else B
#             หมายถึง "ถ้าเงื่อนไขเป็นจริง ใช้ A ถ้าไม่จริง ใช้ B"
#             -> ถ้ามี GPU แสดง "→ พร้อมใช้งาน" ถ้าไม่มี แสดงคำแนะนำวิธีเปิด GPU ในเมนูของ Colab
print("มี GPU หรือไม่:", torch.cuda.is_available(),
      "→ พร้อมใช้งาน" if torch.cuda.is_available() else "→ โปรดเปิดใช้ GPU: Runtime → Change runtime type → T4 GPU แล้วรันเซลล์อีกครั้ง")
# โหลดโมเดล YOLO ที่ผ่านการฝึกล่วงหน้า แล้วเก็บไว้ในตัวแปร model
#   - YOLO(...) : สร้างโมเดลจากไฟล์น้ำหนัก (weights) ที่ระบุ หากยังไม่มีไฟล์ในเครื่องจะดาวน์โหลดให้อัตโนมัติ
#   - "yolo11n.pt" : ไฟล์โมเดล YOLO11 ขนาด n (nano = เล็กและเร็วที่สุด) นามสกุล .pt คือไฟล์ของ PyTorch
#                    โมเดลนี้ฝึกด้วยชุดข้อมูล COCO ซึ่งเป็นภาพวัตถุทั่วไป เช่น คน รถ สุนัข
model = YOLO("yolo11n.pt")   # โมเดล nano ที่ฝึกล่วงหน้าด้วยภาพวัตถุทั่วไป (ชุดข้อมูล COCO ดาวน์โหลดอัตโนมัติ)
# แสดงจำนวนคลาสที่โมเดลรู้จัก และตัวอย่างชื่อคลาส 10 ชื่อแรก
#   - model.names : พจนานุกรม (dict) ที่จับคู่ "หมายเลขคลาส" กับ "ชื่อคลาส" เช่น {0: 'person', 1: 'bicycle', ...}
#                   (dict คือโครงสร้างข้อมูลแบบ {กุญแจ: ค่า})
#   - len(...) : นับจำนวนสมาชิก ในที่นี้คือจำนวนคลาส (COCO มี 80 คลาส)
#   - model.names.values() : ดึงเฉพาะ "ค่า" (ชื่อคลาส) ออกมาจาก dict
#   - list(...) : แปลงเป็นลิสต์ (list = รายการข้อมูลที่เรียงลำดับ เขียนใน [ ])
#   - [:10] : การตัดช่วง (slicing) เอาสมาชิกตำแหน่งแรกถึงตำแหน่งที่ 10 (Python เริ่มนับตำแหน่งที่ 0)
print("โมเดลรู้จักวัตถุ", len(model.names), "คลาส ตัวอย่างเช่น:", list(model.names.values())[:10])

In [ ]:
# ทดสอบการตรวจจับวัตถุในภาพถ่ายทั่วไป เพื่อตรวจสอบว่าระบบทำงานถูกต้อง
# ============================================================================
# 🎯 วัตถุประสงค์ของเซลล์นี้
#   ทดลองใช้โมเดล YOLO ที่ผ่านการฝึกล่วงหน้า (จากเซลล์ก่อนหน้า) ตรวจจับวัตถุในภาพถ่ายถนนตัวอย่าง 1 ภาพ
#   เพื่อยืนยันว่าเครื่องมือทำงานได้ถูกต้องก่อนนำไปใช้กับภาพทางการแพทย์
#   ข้อมูลเข้า : ภาพรถประจำทาง (bus.jpg) จากเว็บไซต์ของ Ultralytics
#   ผลลัพธ์    : 1) ภาพที่มีกรอบล้อมวัตถุ (bounding box) พร้อมชื่อคลาสและค่าความเชื่อมั่น
#               2) รายการข้อความ 1 บรรทัดต่อวัตถุที่ตรวจพบ บอกชื่อวัตถุและค่าความเชื่อมั่น (confidence)
#   ค่าความเชื่อมั่นอยู่ระหว่าง 0 ถึง 1 ยิ่งใกล้ 1 แปลว่าโมเดลยิ่งมั่นใจว่ามีวัตถุนั้นอยู่จริง
# ============================================================================

# ----------------------------------------------------------------------------
# ส่วนที่ 1: ให้โมเดลทำนายภาพ แล้วแสดงภาพที่มีกรอบการตรวจจับ
# ----------------------------------------------------------------------------
# ส่งภาพเข้าโมเดลเพื่อทำนาย (predict) แล้วเก็บผลลัพธ์ไว้ในตัวแปร res
#   - model(...) : การเรียกโมเดลเหมือนเรียกฟังก์ชัน คือสั่งให้โมเดลตรวจจับวัตถุในภาพที่ส่งเข้าไป
#   - "https://ultralytics.com/images/bus.jpg" : ที่อยู่ภาพบนอินเทอร์เน็ต โมเดลจะดาวน์โหลดภาพมาเองอัตโนมัติ
#   - ผลลัพธ์ res เป็นลิสต์ของผลการทำนาย 1 รายการต่อ 1 ภาพ (ที่นี่มี 1 ภาพ จึงใช้ res[0])
res = model("https://ultralytics.com/images/bus.jpg")
# แสดงภาพผลการตรวจจับ (บรรทัดนี้มี 4 คำสั่งเรียงกัน คั่นด้วยเครื่องหมาย ; ซึ่งเทียบเท่ากับการขึ้นบรรทัดใหม่)
#   1) plt.figure(figsize=(8, 6)) : สร้างพื้นที่วาดภาพใหม่ (ไลบรารี matplotlib)
#        - figsize=(8, 6) : ขนาดภาพ กว้าง 8 นิ้ว x สูง 6 นิ้ว
#   2) plt.imshow(...) : วาดภาพ (array ของพิกเซล) ลงในพื้นที่วาดภาพ
#        - res[0] : ผลการทำนายของภาพแรก (ตำแหน่งที่ 0)
#        - .plot() : เมธอดของ ultralytics ที่วาดกรอบ ชื่อคลาส และค่าความเชื่อมั่นลงบนภาพ
#                    ได้ภาพที่เรียงสีแบบ BGR (น้ำเงิน-เขียว-แดง) ตามรูปแบบของไลบรารี OpenCV
#        - [..., ::-1] : กลับลำดับช่องสีจาก BGR เป็น RGB (แดง-เขียว-น้ำเงิน) ซึ่ง matplotlib ใช้
#                        ("..." หมายถึงทุกมิติด้านหน้า และ ::-1 หมายถึงเรียงย้อนกลับในมิติสุดท้าย คือมิติสี)
#                        หากไม่กลับลำดับ สีของภาพจะเพี้ยน (เช่น สีแดงกลายเป็นสีน้ำเงิน)
#   3) plt.axis("off") : ซ่อนแกนและตัวเลขพิกัดรอบภาพ เพื่อให้ดูภาพได้ชัดเจน
#   4) plt.show() : แสดงภาพออกมาใต้เซลล์
plt.figure(figsize=(8, 6)); plt.imshow(res[0].plot()[..., ::-1]); plt.axis("off"); plt.show()

# ----------------------------------------------------------------------------
# ส่วนที่ 2: พิมพ์รายการวัตถุที่ตรวจพบทีละวัตถุ
# ----------------------------------------------------------------------------
# วนซ้ำ (for loop) ผ่านกรอบการตรวจจับทุกกรอบในภาพ
#   - for ตัวแปร in กลุ่มข้อมูล: คือการทำคำสั่งที่ย่อหน้าเข้าไปด้านล่างซ้ำ 1 ครั้งต่อสมาชิก 1 ตัว
#   - res[0].boxes : กลุ่มของกรอบที่โมเดลตรวจพบในภาพแรก
#   - box : ตัวแปรที่แทนกรอบ "ทีละกรอบ" ในแต่ละรอบของการวนซ้ำ
for box in res[0].boxes:     # แสดงหนึ่งบรรทัดต่อวัตถุที่ตรวจพบ
    # พิมพ์ชื่อวัตถุและค่าความเชื่อมั่นของกรอบนี้ ด้วย f-string (ข้อความที่แทรกค่าใน { } ได้)
    #   - box.cls : หมายเลขคลาสของกรอบ (เก็บเป็น tensor ของ PyTorch) จึงแปลงเป็นจำนวนเต็มด้วย int(...)
    #   - model.names[...] : เปิดดูชื่อคลาสจากหมายเลข เช่น 0 -> "person", 5 -> "bus"
    #   - :>10s : รหัสจัดรูปแบบ แสดงข้อความ (s = string) ชิดขวา (>) ในความกว้าง 10 ตัวอักษร ให้ตารางตรงกัน
    #   - box.conf : ค่าความเชื่อมั่นของกรอบ แปลงเป็นทศนิยมด้วย float(...)
    #   - :.2f : แสดงเป็นทศนิยม (f = float) 2 ตำแหน่ง เช่น 0.87
    print(f"{model.names[int(box.cls)]:>10s}  ความเชื่อมั่น (confidence) = {float(box.conf):.2f}")

💡 **คำอธิบายผลลัพธ์**
โมเดลที่ผ่านการฝึกล่วงหน้าวาดกรอบล้อมบุคคลแต่ละคนและรถประจำทาง พร้อมป้ายระบุคลาสและค่าความเชื่อมั่น โมเดลนี้ไม่เคยได้รับการฝึกด้วยภาพทางการแพทย์ จึงรู้จักเฉพาะวัตถุทั่วไป ในขั้นตอนต่อไปจะ *ปรับละเอียด (fine-tune)* โมเดล กล่าวคือ ฝึกโมเดลเพิ่มเติมสำหรับงานใหม่ด้วยชุดข้อมูลขนาดเล็กที่มีป้ายกำกับ เพื่อให้ตรวจหาเนื้องอกสมองในภาพ MRI ได้

### การประมวลผลวิดีโอ

วิดีโอประกอบด้วยภาพนิ่งจำนวนมากที่เรียงต่อกัน เรียกว่า **เฟรม (frame)** โมเดล YOLO จึงประมวลผลทีละเฟรม ในเซลล์ถัดไปจะประมวลผลคลิปวิดีโอถนนความยาวสั้น โดยเลือกเฉพาะทุกเฟรมที่ 5 (`vid_stride=5` เพื่อลดเวลาประมวลผล) และบันทึกเป็นวิดีโอที่มีกรอบการตรวจจับ หลักการเดียวกันนี้ใช้ในระบบตรวจหา polyp หรือรอยโรคแบบเวลาจริง (real-time) ระหว่างการส่องกล้องหรือการตรวจอัลตราซาวนด์ กล่าวคือ โมเดลเดียวกันประมวลผลทุกเฟรมของภาพวิดีโอที่รับเข้ามา

In [ ]:
# ดาวน์โหลดวิดีโอตัวอย่างและตรวจจับวัตถุทีละเฟรม (ประมาณ 30 วินาทีบน GPU)
# ============================================================================
# 🎯 วัตถุประสงค์ของเซลล์นี้
#   สาธิตการใช้โมเดล YOLO กับ "วิดีโอ" ซึ่งก็คือภาพนิ่งจำนวนมากเรียงต่อกัน เรียกว่า เฟรม (frame)
#   โมเดลจะตรวจจับวัตถุทีละเฟรม หลักการเดียวกันนี้ใช้ในระบบตรวจหา polyp ระหว่างส่องกล้องลำไส้ใหญ่
#   หรือการตรวจหารอยโรคแบบเวลาจริง (real-time) ระหว่างอัลตราซาวนด์
#   ขั้นตอน : 1) ดาวน์โหลดคลิปวิดีโอถนนตัวอย่าง
#             2) ให้โมเดลประมวลผลทุกเฟรมที่ 5 และบันทึกเป็นวิดีโอใหม่ที่มีกรอบการตรวจจับ
#             3) เก็บเฟรมตัวอย่างไว้ไม่เกิน 4 เฟรม แล้วแสดงเรียงกันเป็นแถวเดียว
#   ผลลัพธ์ : ข้อความบอกจำนวนเฟรมที่ประมวลผล ตำแหน่งไฟล์วิดีโอผลลัพธ์ และภาพตัวอย่างเฟรมที่มีกรอบ
# ============================================================================

# ----------------------------------------------------------------------------
# ส่วนที่ 1: ดาวน์โหลดวิดีโอตัวอย่าง
# ----------------------------------------------------------------------------
# ดาวน์โหลดไฟล์วิดีโอจากอินเทอร์เน็ตด้วยคำสั่ง wget ของระบบปฏิบัติการ
#   - ! ด้านหน้า : สั่งงานระบบปฏิบัติการ (ไม่ใช่คำสั่ง Python)
#   - wget : โปรแกรมดาวน์โหลดไฟล์จากที่อยู่เว็บ (URL)
#   - -q : (quiet) ไม่ต้องแสดงความคืบหน้าระหว่างดาวน์โหลด
#   - -O sample.mp4 : บันทึกไฟล์ที่ดาวน์โหลดโดยตั้งชื่อว่า sample.mp4
#   - https://github.com/... : ที่อยู่ของคลิปวิดีโอตัวอย่าง (คน จักรยาน รถยนต์) จากคลังของ Intel
!wget -q -O sample.mp4 https://github.com/intel-iot-devkit/sample-videos/raw/master/person-bicycle-car-detection.mp4

# ----------------------------------------------------------------------------
# ส่วนที่ 2: ให้โมเดลตรวจจับวัตถุทีละเฟรม และเก็บเฟรมตัวอย่างไว้แสดง
# ----------------------------------------------------------------------------
# สร้างลิสต์ว่าง (list เปล่า เขียนเป็น [ ]) ชื่อ frames ไว้เก็บเฟรมตัวอย่างที่จะนำมาแสดงภายหลัง
frames = []
# วนซ้ำผ่านผลการทำนายของแต่ละเฟรม
#   - model.predict(...) : เมธอดของ ultralytics สำหรับให้โมเดลทำนาย (ตรวจจับวัตถุ) บนภาพหรือวิดีโอ
#       - source="sample.mp4" : แหล่งข้อมูลที่จะประมวลผล คือไฟล์วิดีโอที่เพิ่งดาวน์โหลด
#       - save=True : บันทึกผลลัพธ์ (วิดีโอที่วาดกรอบแล้ว) ลงโฟลเดอร์ runs/detect/... โดยอัตโนมัติ
#       - vid_stride=5 : เลือกประมวลผลเพียงทุกเฟรมที่ 5 ของวิดีโอ เพื่อลดเวลาประมวลผล
#       - stream=True : ส่งผลลัพธ์ออกมาทีละเฟรม (generator) แทนการเก็บทุกเฟรมไว้ในหน่วยความจำพร้อมกัน
#                       ช่วยป้องกันหน่วยความจำเต็มเมื่อวิดีโอยาว
#       - verbose=False : ไม่ต้องพิมพ์ข้อความรายละเอียดของแต่ละเฟรม
#   - enumerate(...) : ฟังก์ชันพื้นฐานของ Python ที่ให้ "ลำดับที่" (เริ่มจาก 0) คู่กับสมาชิกแต่ละตัว
#   - i : ลำดับที่ของเฟรมที่ประมวลผล (0, 1, 2, ...)
#   - r : ผลการทำนายของเฟรมนั้น (Results object ของ ultralytics)
for i, r in enumerate(model.predict(source="sample.mp4", save=True, vid_stride=5, stream=True, verbose=False)):
    # ตรวจสอบเงื่อนไขด้วย if (ถ้าเงื่อนไขเป็นจริง จึงทำคำสั่งที่ย่อหน้าเข้าไปด้านล่าง)
    #   - i % 40 == 0 : เครื่องหมาย % คือเศษจากการหาร หากเศษเป็น 0 แปลว่า i หาร 40 ลงตัว
    #                   คือเลือกเฉพาะเฟรมลำดับที่ 0, 40, 80, ... (== คือการเปรียบเทียบว่าเท่ากันหรือไม่)
    #   - and : ต้องเป็นจริงทั้งสองเงื่อนไข
    #   - len(frames) < 4 : ยังเก็บเฟรมไว้ไม่ถึง 4 เฟรม (จำกัดจำนวนภาพที่จะแสดง)
    if i % 40 == 0 and len(frames) < 4:          # เก็บเฟรมที่มีกรอบการตรวจจับไว้จำนวนหนึ่งเพื่อแสดงผล
        # วาดกรอบการตรวจจับลงบนเฟรม แล้วเพิ่มเข้าไปท้ายลิสต์ frames
        #   - r.plot() : วาดกรอบ ชื่อคลาส และค่าความเชื่อมั่นลงบนเฟรม (ได้ภาพเรียงสีแบบ BGR)
        #   - [..., ::-1] : กลับลำดับช่องสีจาก BGR เป็น RGB เพื่อให้ matplotlib แสดงสีถูกต้อง
        #   - frames.append(...) : เมธอดของลิสต์ สำหรับเพิ่มสมาชิกใหม่ต่อท้าย
        frames.append(r.plot()[..., ::-1])
    # จดจำโฟลเดอร์ที่ ultralytics ใช้บันทึกวิดีโอผลลัพธ์ (r.save_dir) ไว้ในตัวแปร save_dir เพื่อแจ้งผู้เรียนภายหลัง
    save_dir = r.save_dir
# แสดงสรุปผลหลังวนซ้ำครบทุกเฟรม (บรรทัดนี้ไม่ย่อหน้า จึงทำงานครั้งเดียวหลังจบ for)
#   - {i + 1} : จำนวนเฟรมที่ประมวลผล (บวก 1 เพราะ i เริ่มนับจาก 0)
#   - {save_dir} : ตำแหน่งโฟลเดอร์ที่เก็บวิดีโอที่มีกรอบการตรวจจับ
print(f"ประมวลผลแล้ว {i + 1} เฟรม บันทึกวิดีโอที่มีกรอบการตรวจจับไว้ที่: {save_dir}  (ดาวน์โหลดได้จากแผง 📁 Files ด้านซ้าย)")

# ----------------------------------------------------------------------------
# ส่วนที่ 3: แสดงเฟรมตัวอย่างเรียงกันในแถวเดียว
# ----------------------------------------------------------------------------
# สร้างพื้นที่วาดภาพที่แบ่งเป็นช่องย่อยหลายช่อง ด้วย plt.subplots(...) ของ matplotlib
#   - 1 : จำนวนแถว = 1 แถว
#   - len(frames) : จำนวนคอลัมน์ = จำนวนเฟรมที่เก็บไว้ (1 ช่องต่อ 1 เฟรม)
#   - figsize=(4 * len(frames), 3) : ขนาดภาพรวม กว้าง 4 นิ้วต่อเฟรม สูง 3 นิ้ว
#   - squeeze=False : ให้ผลลัพธ์ axes เป็นตาราง 2 มิติ (แถว x คอลัมน์) เสมอ แม้มีเพียงช่องเดียว
#   - ผลลัพธ์มี 2 ค่า: fig (ภาพรวมทั้งหมด) และ axes (ช่องย่อยแต่ละช่อง) จึงรับด้วย fig, axes =
fig, axes = plt.subplots(1, len(frames), figsize=(4 * len(frames), 3), squeeze=False)
# วนซ้ำจับคู่ "ช่องย่อย" กับ "เฟรม" ทีละคู่
#   - axes[0] : ช่องย่อยทั้งหมดในแถวแรก (แถวที่ 0)
#   - zip(A, B) : ฟังก์ชันพื้นฐานของ Python ที่จับคู่สมาชิกตัวที่ 1 ของ A กับตัวที่ 1 ของ B, ตัวที่ 2 กับตัวที่ 2, ...
#   - ax : ช่องย่อยในรอบนั้น, f : เฟรมในรอบนั้น
for ax, f in zip(axes[0], frames):
    # วาดเฟรมลงในช่องย่อย (ax.imshow) แล้วซ่อนแกนพิกัด (ax.axis("off")) — 2 คำสั่งคั่นด้วย ;
    ax.imshow(f); ax.axis("off")
# จัดระยะห่างของช่องย่อยไม่ให้ทับกัน (plt.tight_layout) แล้วแสดงภาพออกมาใต้เซลล์ (plt.show)
plt.tight_layout(); plt.show()

💡 **คำอธิบายผลลัพธ์**
โมเดลประมวลผลแต่ละเฟรมที่เลือกไว้แยกจากกัน วาดกรอบการตรวจจับลงบนเฟรม แล้วนำเฟรมมาเรียงต่อกันเป็นไฟล์วิดีโอใหม่ ในระบบวิดีโอทางคลินิก เช่น การส่องกล้องลำไส้ใหญ่ แพทย์จะเห็นกรอบเหล่านี้ปรากฏบนจอภาพแบบเวลาจริงระหว่างการเคลื่อนกล้อง

## การปรับละเอียดโมเดลด้วยข้อมูลทางการแพทย์ (ภาพ MRI เนื้องอกสมอง)

ส่วนนี้ฝึกโมเดลสำหรับงานทางการแพทย์ด้วยชุดข้อมูลสาธารณะ **brain-tumor** ซึ่งมาพร้อมกับ Ultralytics ประกอบด้วยภาพ MRI **ชุดฝึก 893 ภาพ** และ **ชุดตรวจสอบ 223 ภาพ** โดยผู้เชี่ยวชาญวาดกรอบล้อมบริเวณรอยโรคและกำหนดป้ายของแต่ละกรอบเป็น `negative` หรือ `positive` (สองคลาสตามที่ผู้จัดทำชุดข้อมูลกำหนด) ชุดข้อมูลมีขนาดเพียง 4 MB และดาวน์โหลดอัตโนมัติ ขั้นตอนเดียวกันนี้สามารถประยุกต์ใช้กับการตรวจหา polyp ก้อนในปอด (lung nodule) หรือกระดูกหัก

- **ภาพชุดฝึก (training images)** — ภาพที่โมเดลใช้เรียนรู้
- **ภาพชุดตรวจสอบ (validation images)** — ภาพที่แยกไว้ไม่ใช้ฝึก เพื่อวัดประสิทธิภาพของโมเดลกับภาพ MRI ที่โมเดลไม่เคยเห็น
- **Epoch** — การฝึกโมเดลด้วยภาพชุดฝึกครบทุกภาพหนึ่งรอบ จำนวน epoch ที่มากขึ้นทำให้โมเดลเรียนรู้ได้มากขึ้น แต่ใช้เวลานานขึ้น

(หมายเหตุทางเทคนิค: ไฟล์ `brain-tumor.yaml` ระบุรายละเอียดของชุดข้อมูล ป้ายกำกับจัดเก็บในรูปแบบ YOLO `class cx cy w h` ซึ่งปรับสเกลเป็นช่วง 0–1)

> ⏱️ การฝึกโมเดลเป็นขั้นตอนเดียวที่ใช้เวลานาน ประมาณ 10–20 วินาทีต่อ epoch บน GPU T4 และหลายนาทีต่อ epoch บน CPU

In [ ]:
# ⭐ Exercise 1 — แบบฝึกหัดที่ 1: ปรับละเอียดโมเดล YOLO ด้วยภาพ MRI เนื้องอกสมอง
# หน้าที่ของเซลล์: เริ่มจากโมเดล nano ที่ฝึกล่วงหน้า แล้วฝึกเพิ่มเติมด้วยภาพ MRI ชุดฝึก 893 ภาพ
# สิ่งที่ต้องแก้ไข: แทนที่ ... ด้วยจำนวน epoch (จำนวนรอบการฝึก)
# คำแนะนำ: EPOCHS = 10 (ประมาณ 3 นาทีบน GPU T4) หากไม่มี GPU ให้ใช้ EPOCHS = 3 เพื่อลดเวลาการฝึก
# ============================================================================
# 🎯 วัตถุประสงค์ของเซลล์นี้
#   ปรับละเอียด (fine-tune) โมเดล YOLO ที่เคยเรียนรู้เฉพาะวัตถุทั่วไป ให้ตรวจหาเนื้องอกสมองในภาพ MRI ได้
#   โดยฝึกเพิ่มเติมด้วยชุดข้อมูล brain-tumor ซึ่งผู้เชี่ยวชาญวาดกรอบรอยโรคไว้แล้ว (2 คลาส: negative / positive)
#   ข้อมูลเข้า : ภาพ MRI ชุดฝึก (training images) 893 ภาพ พร้อมกรอบของผู้เชี่ยวชาญ
#   ผลลัพธ์    : โมเดลที่ฝึกแล้วเก็บในตัวแปร det และระหว่างฝึกจะเห็นตารางความคืบหน้าของแต่ละ epoch
#                (ค่า loss = ความคลาดเคลื่อนของโมเดล ควรลดลง และค่า mAP = คุณภาพการตรวจจับ ควรเพิ่มขึ้น)
#   ผู้เรียนต้องกรอกจำนวน epoch ในตำแหน่ง ... ก่อนรันเซลล์
# ============================================================================

# กำหนดจำนวนรอบการฝึก (epoch) เก็บไว้ในตัวแปร EPOCHS
#   - 1 epoch = โมเดลได้เรียนรู้จากภาพชุดฝึกครบทุกภาพ 1 รอบ
#   - ... (จุด 3 จุด) คือ "ช่องว่าง" ที่ผู้เรียนต้องแทนที่ด้วยตัวเลขจำนวนเต็ม (ดูคำแนะนำด้านบน)
#   - จำนวนรอบมากขึ้นมักได้โมเดลที่ดีขึ้น แต่ใช้เวลาฝึกนานขึ้น
EPOCHS = ...

# สร้างโมเดลใหม่จากโมเดล nano ที่ฝึกล่วงหน้า (yolo11n.pt) แล้วเก็บไว้ในตัวแปร det (ย่อมาจาก detector)
#   - เริ่มจากโมเดลที่ฝึกล่วงหน้าแทนการเริ่มจากศูนย์ เพราะโมเดลรู้จักรูปทรงและขอบของวัตถุแล้ว
#     จึงเรียนรู้งานใหม่ได้เร็วด้วยข้อมูลจำนวนน้อย
det = YOLO("yolo11n.pt")
# สั่งฝึกโมเดลด้วยเมธอด .train(...) ของ ultralytics (คำสั่งนี้เขียนหลายบรรทัด โดยแต่ละพารามิเตอร์อยู่คนละบรรทัด)
det.train(
    # data : ไฟล์ตั้งค่าชุดข้อมูล (.yaml) ซึ่งระบุตำแหน่งภาพชุดฝึก/ชุดตรวจสอบและชื่อคลาส
    #        "brain-tumor.yaml" มาพร้อม ultralytics และจะดาวน์โหลดภาพให้อัตโนมัติเมื่อใช้ครั้งแรก
    data="brain-tumor.yaml",   # มาพร้อมกับ ultralytics และดาวน์โหลดอัตโนมัติ
    # epochs : จำนวนรอบการฝึก ใช้ค่าจากตัวแปร EPOCHS ที่ผู้เรียนกำหนดด้านบน
    epochs=EPOCHS,
    # imgsz : ขนาดภาพที่ป้อนให้โมเดล (หน่วยพิกเซล) ภาพจะถูกปรับขนาดเป็น 416 x 416 ก่อนฝึก
    imgsz=416,                 # ปรับขนาดภาพเป็น 416 x 416 พิกเซล
    # batch : จำนวนภาพที่โมเดลประมวลผลพร้อมกันก่อนปรับพารามิเตอร์ 1 ครั้ง (ชุดย่อย = batch)
    batch=16,                  # ฝึกครั้งละ 16 ภาพ
)

In [ ]:
# @title ✅ เฉลย — หากไม่สามารถทำแบบฝึกหัดได้ ให้รันเซลล์นี้ (ดับเบิลคลิกเพื่อแสดงโค้ด) { display-mode: "form" }
# ============================================================================
# 🎯 วัตถุประสงค์ของเซลล์นี้ (เฉลยแบบฝึกหัดที่ 1)
#   ปรับละเอียด (fine-tune) โมเดล YOLO nano ด้วยภาพ MRI เนื้องอกสมองชุดฝึก 893 ภาพ
#   เฉลยนี้เลือกจำนวนรอบการฝึก (epoch) ให้อัตโนมัติ: 10 รอบเมื่อมี GPU และ 3 รอบเมื่อใช้ CPU
#   (CPU ประมวลผลช้ากว่า GPU มาก จึงลดจำนวนรอบเพื่อให้ฝึกเสร็จในเวลาที่เหมาะสม)
#   ผลลัพธ์ : โมเดลที่ฝึกแล้วเก็บในตัวแปร det พร้อมตารางความคืบหน้าของแต่ละ epoch
# ============================================================================

# กำหนดจำนวนรอบการฝึกด้วยนิพจน์เงื่อนไขแบบบรรทัดเดียว (A if เงื่อนไข else B)
#   - torch.cuda.is_available() : ตรวจว่ามี GPU หรือไม่ (True/False)
#   - ถ้ามี GPU -> EPOCHS = 10 (ประมาณ 3 นาทีบน GPU T4)
#   - ถ้าไม่มี  -> EPOCHS = 3
EPOCHS = 10 if torch.cuda.is_available() else 3   # หากใช้ CPU ให้ฝึกเพียงไม่กี่รอบ

# สร้างโมเดลจากโมเดล nano ที่ฝึกล่วงหน้า (yolo11n.pt) แล้วเก็บไว้ในตัวแปร det (ย่อมาจาก detector)
#   เริ่มจากโมเดลที่ฝึกล่วงหน้า เพราะโมเดลรู้จักลักษณะพื้นฐานของภาพ (ขอบ รูปทรง) แล้ว จึงเรียนรู้งานใหม่ได้เร็ว
det = YOLO("yolo11n.pt")
# สั่งฝึกโมเดลด้วยเมธอด .train(...) ของ ultralytics (คำสั่งเดียวที่เขียนหลายบรรทัด)
det.train(
    # data : ไฟล์ตั้งค่าชุดข้อมูล (.yaml) ระบุตำแหน่งภาพชุดฝึก/ชุดตรวจสอบและชื่อคลาส (ดาวน์โหลดอัตโนมัติ)
    data="brain-tumor.yaml",   # มาพร้อมกับ ultralytics และดาวน์โหลดอัตโนมัติ
    # epochs : จำนวนรอบการฝึก (1 epoch = เรียนรู้จากภาพชุดฝึกครบทุกภาพ 1 รอบ)
    epochs=EPOCHS,
    # imgsz : ขนาดภาพที่ป้อนให้โมเดล ภาพจะถูกปรับขนาดเป็น 416 x 416 พิกเซล
    imgsz=416,                 # ปรับขนาดภาพเป็น 416 x 416 พิกเซล
    # batch : จำนวนภาพที่ประมวลผลพร้อมกันก่อนปรับพารามิเตอร์ของโมเดล 1 ครั้ง
    batch=16,                  # ฝึกครั้งละ 16 ภาพ
)

💡 **คำอธิบายผลลัพธ์**
ในแต่ละ epoch โมเดลทำนายกรอบบนภาพ MRI ชุดฝึก เปรียบเทียบกับกรอบของผู้เชี่ยวชาญ และปรับพารามิเตอร์เพียงเล็กน้อยเพื่อลดความคลาดเคลื่อน เมื่อจบแต่ละ epoch ระบบจะวัดผลกับภาพชุดตรวจสอบ กราฟการฝึกด้านล่างแสดงผลดังกล่าว โดยค่า **loss** (ค่าความคลาดเคลื่อนของโมเดล) ควร *ลดลง* และค่า **mAP** (คุณภาพการตรวจจับ ซึ่งอธิบายในแบบฝึกหัดที่ 2) ควร *เพิ่มขึ้น* (ชื่อกราฟเป็นภาษาอังกฤษตามที่ Ultralytics สร้างขึ้น)

In [ ]:
# กราฟการฝึก (Ultralytics บันทึกไว้โดยอัตโนมัติ)
# ============================================================================
# 🎯 วัตถุประสงค์ของเซลล์นี้
#   แสดง "กราฟการฝึก" ที่ Ultralytics สร้างและบันทึกไว้เป็นไฟล์ results.png ระหว่างการฝึกโมเดล
#   เพื่อให้ผู้เรียนเห็นว่าโมเดลเรียนรู้ดีขึ้นในแต่ละ epoch หรือไม่
#   - ค่า loss (ความคลาดเคลื่อนของโมเดล) ควรลดลง
#   - ค่า mAP (คุณภาพการตรวจจับ) ควรเพิ่มขึ้น
#   ผลลัพธ์ : ภาพกราฟหลายช่อง (ชื่อกราฟเป็นภาษาอังกฤษ) หรือข้อความแจ้งหากไม่พบไฟล์
# ============================================================================

# นำเข้าไลบรารี os (operating system) สำหรับจัดการชื่อไฟล์/โฟลเดอร์ และตรวจสอบว่ามีไฟล์อยู่หรือไม่
import os
# นำเข้าเครื่องมือ 2 ตัวจากไลบรารี IPython (ระบบเบื้องหลังของ Jupyter/Colab)
#   - Image   : ใช้เปิดไฟล์ภาพเพื่อแสดงในโน้ตบุ๊ก
#   - display : ใช้แสดงผลวัตถุ (เช่น ภาพหรือตาราง) ออกมาใต้เซลล์
from IPython.display import Image, display
# สร้างตำแหน่งเต็มของไฟล์กราฟ แล้วเก็บไว้ในตัวแปร curves
#   - det.trainer.save_dir : โฟลเดอร์ที่ ultralytics บันทึกผลการฝึกครั้งล่าสุด (เช่น runs/detect/train)
#   - "results.png" : ชื่อไฟล์กราฟการฝึกที่ ultralytics สร้างให้
#   - os.path.join(...) : ต่อชื่อโฟลเดอร์กับชื่อไฟล์ให้เป็นตำแหน่งไฟล์ที่ถูกต้อง (ใส่เครื่องหมาย / ให้เอง)
curves = os.path.join(det.trainer.save_dir, "results.png")
# แสดงภาพกราฟถ้ามีไฟล์อยู่ ถ้าไม่มีให้พิมพ์ข้อความแจ้ง (นิพจน์เงื่อนไขแบบบรรทัดเดียว A if เงื่อนไข else B)
#   - os.path.exists(curves) : ตรวจว่ามีไฟล์ตามตำแหน่ง curves อยู่จริงหรือไม่ (True/False)
#   - display(Image(curves, width=1000)) : เปิดไฟล์ภาพแล้วแสดงใต้เซลล์
#       - curves : ตำแหน่งไฟล์ภาพ
#       - width=1000 : ความกว้างของภาพที่แสดง 1000 พิกเซล
#   - print("ไม่พบไฟล์ results.png ใน", ...) : ข้อความแจ้งเมื่อไม่พบไฟล์ พร้อมบอกโฟลเดอร์ที่ค้นหา
display(Image(curves, width=1000)) if os.path.exists(curves) else print("ไม่พบไฟล์ results.png ใน", det.trainer.save_dir)

In [ ]:
# ⭐ Exercise 2 — แบบฝึกหัดที่ 2: ตรวจสอบโมเดลและอ่านค่าตัวชี้วัด
# หน้าที่ของเซลล์: ให้โมเดลที่ฝึกแล้วทำนายภาพ MRI ชุดตรวจสอบ 223 ภาพ และให้คะแนนเทียบกับกรอบของผู้เชี่ยวชาญ
# สิ่งที่ต้องแก้ไข: แทนที่ ... แต่ละตำแหน่งด้วยชื่อตัวชี้วัดตามคำแนะนำ
# คำแนะนำ: ค่า mAP50 รวม คือ  metrics.box.map50   และค่า mAP50-95 คือ  metrics.box.map
# ============================================================================
# 🎯 วัตถุประสงค์ของเซลล์นี้
#   ประเมินผล (validate) โมเดลที่ฝึกแล้ว ด้วยภาพ MRI ชุดตรวจสอบ (validation images) 223 ภาพที่โมเดลไม่เคยเห็นตอนฝึก
#   แล้วอ่านค่าตัวชี้วัดคุณภาพการตรวจจับ ได้แก่
#   - mAP50    : ค่าเฉลี่ยความแม่นยำ (mean Average Precision) เมื่อถือว่ากรอบถูกต้องที่ IoU >= 0.5
#   - mAP50-95 : แบบเข้มงวดกว่า เฉลี่ยตั้งแต่ IoU 0.5 ถึง 0.95 (กรอบต้องวางตำแหน่งแม่นยำมาก)
#   - AP50 แยกตามคลาส (negative / positive)
#   (IoU = Intersection over Union คือระดับการซ้อนทับของกรอบโมเดลกับกรอบผู้เชี่ยวชาญ ตั้งแต่ 0 ถึง 1)
#   ผลลัพธ์ : ตารางผลการประเมินจาก ultralytics และตัวเลขสรุปที่พิมพ์ออกมา (ค่ายิ่งใกล้ 1 ยิ่งดี)
#   ผู้เรียนต้องกรอกตำแหน่ง ... ให้ครบก่อนรันเซลล์
# ============================================================================

# ประเมินผลโมเดลด้วยเมธอด .val(...) ของ ultralytics แล้วเก็บผลตัวชี้วัดทั้งหมดไว้ในตัวแปร metrics
#   - data="brain-tumor.yaml" : ใช้ชุดข้อมูลเดียวกับตอนฝึก ซึ่งจะประเมินบนภาพชุดตรวจสอบ (val)
#   - imgsz=416 : ขนาดภาพ 416 x 416 พิกเซล ควรเท่ากับขนาดที่ใช้ตอนฝึก
metrics = det.val(data="brain-tumor.yaml", imgsz=416)
# ดึงค่า mAP50 (ภาพรวมทุกคลาส) จากผลการประเมิน มาเก็บในตัวแปร map50
#   - ... คือช่องว่างที่ผู้เรียนต้องแทนที่ด้วยชื่อตัวชี้วัด (ดูคำแนะนำในบรรทัด # TODO และส่วนหัวของเซลล์)
map50    = ...      # TODO: metrics.box.map50
# ดึงค่า mAP50-95 มาเก็บในตัวแปร map50_95
#   - ... คือช่องว่างที่ผู้เรียนต้องแทนที่ (ดูคำแนะนำในบรรทัด # TODO)
map50_95 = ...      # TODO: metrics.box.map

# พิมพ์ค่า mAP50 ด้วย f-string
#   - {map50:.3f} : แทรกค่า map50 แสดงเป็นทศนิยม 3 ตำแหน่ง เช่น 0.512
print(f"mAP50   : {map50:.3f}")
# พิมพ์ค่า mAP50-95 เป็นทศนิยม 3 ตำแหน่งในลักษณะเดียวกัน
print(f"mAP50-95: {map50_95:.3f}")
# วนซ้ำผ่านทุกคลาสที่มีผลการประเมิน เพื่อพิมพ์ค่า AP50 แยกตามคลาส
#   - metrics.box.ap_class_index : ลิสต์หมายเลขคลาสที่มีผลการประเมิน (เช่น 0 = negative, 1 = positive)
#   - enumerate(...) : ให้ลำดับที่ i (0, 1, ...) คู่กับหมายเลขคลาส c
for i, c in enumerate(metrics.box.ap_class_index):          # ค่า AP50 แยกตามคลาส
    # พิมพ์ชื่อคลาสและค่า AP50 ของคลาสนั้น
    #   - det.names[int(c)] : แปลงหมายเลขคลาสเป็นจำนวนเต็ม แล้วเปิดดูชื่อคลาสจาก dict det.names
    #   - metrics.box.ap50[i] : ค่า AP50 ของคลาสลำดับที่ i
    #   - :.3f : แสดงเป็นทศนิยม 3 ตำแหน่ง
    print(f"  AP50 ของคลาส '{det.names[int(c)]}': {metrics.box.ap50[i]:.3f}")

In [ ]:
# @title ✅ เฉลย — หากไม่สามารถทำแบบฝึกหัดได้ ให้รันเซลล์นี้ (ดับเบิลคลิกเพื่อแสดงโค้ด) { display-mode: "form" }
# ============================================================================
# 🎯 วัตถุประสงค์ของเซลล์นี้ (เฉลยแบบฝึกหัดที่ 2)
#   ประเมินผลโมเดลที่ฝึกแล้วบนภาพ MRI ชุดตรวจสอบ 223 ภาพ แล้วอ่านค่าตัวชี้วัด
#   - mAP50    : ค่าเฉลี่ยความแม่นยำทุกคลาส เมื่อถือว่ากรอบถูกต้องที่ IoU >= 0.5
#   - mAP50-95 : แบบเข้มงวด เฉลี่ยตั้งแต่ IoU 0.5 ถึง 0.95
#   - AP50 แยกตามคลาส (negative / positive)
#   ผลลัพธ์ : ตารางผลการประเมินจาก ultralytics และตัวเลขสรุป (ค่ายิ่งใกล้ 1 ยิ่งดี)
# ============================================================================

# ประเมินผลโมเดลด้วยเมธอด .val(...) แล้วเก็บผลตัวชี้วัดไว้ในตัวแปร metrics
#   - data="brain-tumor.yaml" : ชุดข้อมูลเดียวกับตอนฝึก (ประเมินบนภาพชุดตรวจสอบ)
#   - imgsz=416 : ขนาดภาพ 416 x 416 พิกเซล เท่ากับตอนฝึก
metrics = det.val(data="brain-tumor.yaml", imgsz=416)
# ดึงค่า mAP50 ภาพรวม: metrics.box คือกลุ่มตัวชี้วัดของกรอบ และ .map50 คือค่า mAP ที่ IoU 0.5
map50    = metrics.box.map50
# ดึงค่า mAP50-95: .map คือค่า mAP เฉลี่ยตั้งแต่ IoU 0.5 ถึง 0.95
map50_95 = metrics.box.map

# พิมพ์ค่า mAP50 เป็นทศนิยม 3 ตำแหน่ง (:.3f ใน f-string)
print(f"mAP50   : {map50:.3f}")
# พิมพ์ค่า mAP50-95 เป็นทศนิยม 3 ตำแหน่ง
print(f"mAP50-95: {map50_95:.3f}")
# วนซ้ำผ่านทุกคลาสที่มีผลการประเมิน
#   - metrics.box.ap_class_index : ลิสต์หมายเลขคลาสที่มีผลการประเมิน
#   - enumerate(...) : ให้ลำดับที่ i คู่กับหมายเลขคลาส c
for i, c in enumerate(metrics.box.ap_class_index):          # ค่า AP50 แยกตามคลาส
    # พิมพ์ชื่อคลาส (det.names[int(c)]) และค่า AP50 ของคลาสนั้น (metrics.box.ap50[i]) เป็นทศนิยม 3 ตำแหน่ง
    print(f"  AP50 ของคลาส '{det.names[int(c)]}': {metrics.box.ap50[i]:.3f}")

💡 **คำอธิบายผลลัพธ์**
- **IoU** (Intersection over Union) วัดระดับการซ้อนทับระหว่างกรอบที่โมเดลทำนายกับกรอบของผู้เชี่ยวชาญ โดย 1 หมายถึงซ้อนทับกันสมบูรณ์ และ 0 หมายถึงไม่ซ้อนทับกันเลย การทำนายจะนับเป็น *การตรวจพบที่ถูกต้อง* เมื่อ IoU ≥ 0.5 **และ** คลาสถูกต้อง
- **AP50** (Average Precision ที่ IoU 0.5) สรุปประสิทธิภาพของแต่ละคลาส ทั้งด้านความไว (sensitivity / recall) ในการตรวจพบรอยโรค และด้านค่าพยากรณ์บวก (precision / PPV) ในการหลีกเลี่ยงการแจ้งเตือนผิด โดยพิจารณาครอบคลุม *ทุก* ระดับเกณฑ์ความเชื่อมั่น ค่า 1.0 หมายถึงสมบูรณ์
- **mAP50** คือค่าเฉลี่ยของ AP50 ทุกคลาส ส่วน **mAP50-95** มีความเข้มงวดมากกว่า เนื่องจากเฉลี่ยตั้งแต่ IoU 0.5 ถึง 0.95 กรอบจึงต้องมีตำแหน่งที่แม่นยำสูง

เมื่อฝึกเพียงไม่กี่ epoch โมเดล nano มักได้ค่า mAP50 ในระดับปานกลาง ส่วนถัดไปจึงให้ผู้เรียนตรวจพินิจความคลาดเคลื่อนของโมเดลด้วยตนเอง

✍️ **คำถามที่ 1** โปรดระบุว่าคลาสใด (`negative` หรือ `positive`) มีค่า AP50 ต่ำกว่า และจงเสนอสมมติฐานหนึ่งข้อเพื่ออธิบายสาเหตุ (เช่น จำนวนตัวอย่างในชุดฝึกน้อยกว่า รอยโรคมีขนาดเล็กหรือมองเห็นได้ไม่ชัด หรือป้ายกำกับของผู้เชี่ยวชาญไม่สม่ำเสมอ)

✍️ **คำตอบ:** 

## การตรวจพินิจผลการตรวจจับ

ตัวเลขสรุปเพียงค่าเดียว เช่น mAP ไม่ได้แสดง *ลักษณะ* ของความคลาดเคลื่อน ส่วนนี้จึงแสดงภาพ MRI ชุดตรวจสอบที่วาด **ทั้ง** กรอบของผู้เชี่ยวชาญและกรอบของโมเดลบนภาพเดียวกัน

เซลล์เครื่องมือช่วยด้านล่างให้โมเดลทำนายภาพ MRI ชุดตรวจสอบทั้ง 223 ภาพเพียงครั้งเดียว (โดยเก็บแม้แต่กรอบที่มีความเชื่อมั่นต่ำ) และสร้างเครื่องมือ 2 รายการ ได้แก่
- `show_detections(threshold, start=0)` — แสดงภาพ MRI 8 ภาพ โดยแสดงเฉพาะกรอบของโมเดลที่มีความเชื่อมั่น ≥ `threshold`
- `count_errors(thresholds)` — นับจำนวนการตรวจพบที่ถูกต้อง การตรวจไม่พบ และการแจ้งเตือนผิด ตามเกณฑ์ตัดสิน (threshold) หนึ่งค่าหรือหลายค่า

ผู้เรียนไม่จำเป็นต้องอ่านโค้ดในเซลล์นี้ ให้รันเซลล์เท่านั้น

In [ ]:
# @title 🔧 เครื่องมือช่วยตรวจพินิจผลการตรวจจับ — ให้รันเซลล์นี้ { display-mode: "form" }
# ============================================================================
# 🎯 วัตถุประสงค์ของเซลล์นี้
#   เซลล์นี้เตรียม "เครื่องมือช่วย" สำหรับตรวจพินิจ (ดูด้วยตา) ว่าโมเดลตรวจจับถูกหรือผิดอย่างไร
#   เพราะตัวเลขสรุปเพียงค่าเดียว เช่น mAP ไม่ได้บอก "ลักษณะ" ของความคลาดเคลื่อน
#   ขั้นตอนหลัก:
#   1) หาตำแหน่งโฟลเดอร์ภาพ MRI ชุดตรวจสอบ (val) และโฟลเดอร์ป้ายกำกับ (labels) ของผู้เชี่ยวชาญ
#   2) สร้างฟังก์ชันย่อย
#        - load_expert_boxes(...) : อ่านกรอบรอยโรคที่ผู้เชี่ยวชาญวาดไว้ในภาพหนึ่งภาพ
#        - iou(a, b)              : คำนวณระดับการซ้อนทับ (IoU) ของกรอบ 2 กรอบ
#        - judge(...)             : ตัดสินว่ากรอบของโมเดลแต่ละกรอบเป็น hit / wrong label / false alarm
#   3) ให้โมเดลทำนายภาพชุดตรวจสอบทุกภาพ "เพียงครั้งเดียว" และเก็บทุกกรอบไว้ (แม้ความเชื่อมั่นต่ำ)
#      เพื่อนำไปกรองตามเกณฑ์ตัดสิน (threshold) ต่าง ๆ ภายหลังได้อย่างรวดเร็ว
#   4) สร้างเครื่องมือหลัก 2 ตัวที่ผู้เรียนจะใช้ในแบบฝึกหัดถัดไป
#        - show_detections(threshold, start=0) : แสดงภาพ MRI พร้อมกรอบของผู้เชี่ยวชาญและของโมเดล
#        - count_errors([thresholds])          : นับจำนวนการตรวจพบถูกต้อง ตรวจไม่พบ และแจ้งเตือนผิด
#   ผลลัพธ์ : ข้อความสรุปจำนวนกรอบ และข้อความยืนยันว่าเครื่องมือพร้อมใช้งาน
#   ผู้เรียนไม่จำเป็นต้องแก้ไขโค้ด เพียงรันเซลล์นี้ (คำอธิบายด้านล่างมีไว้สำหรับผู้ที่สนใจ)
# ============================================================================

# ----------------------------------------------------------------------------
# ส่วนที่ 1: นำเข้าไลบรารีที่จำเป็น
# ----------------------------------------------------------------------------
# นำเข้าไลบรารี 2 ตัวพร้อมตั้งชื่อย่อ
#   - numpy as np  : ไลบรารีคำนวณตัวเลขและตาราง (array)
#   - pandas as pd : ไลบรารีจัดการข้อมูลแบบตาราง (DataFrame) ใช้สร้างตารางสรุปใน count_errors
import numpy as np, pandas as pd
# นำเข้า Path จากไลบรารี pathlib : เครื่องมือจัดการตำแหน่งไฟล์/โฟลเดอร์ เช่น ต่อชื่อโฟลเดอร์ด้วยเครื่องหมาย /
from pathlib import Path
# นำเข้า Image จากไลบรารี PIL (Pillow ใช้เปิด/จัดการไฟล์ภาพ) แล้วตั้งชื่อใหม่ว่า PILImage
#   - ตั้งชื่อใหม่ด้วย as เพื่อไม่ให้ชนกับ Image ของ IPython ที่นำเข้าไว้ในเซลล์กราฟการฝึก
from PIL import Image as PILImage
# นำเข้าโมดูล patches ของ matplotlib : ใช้วาดรูปทรง เช่น กรอบสี่เหลี่ยม (Rectangle) ลงบนภาพ
import matplotlib.patches as patches
# นำเข้าฟังก์ชัน check_det_dataset ของ ultralytics : อ่านไฟล์ตั้งค่าชุดข้อมูล (.yaml)
#   ตรวจว่ามีข้อมูลในเครื่องแล้ว (ดาวน์โหลดให้หากยังไม่มี) และคืนค่าเป็น dict ที่บอกตำแหน่งโฟลเดอร์ต่าง ๆ
from ultralytics.data.utils import check_det_dataset

# ----------------------------------------------------------------------------
# ส่วนที่ 2: หาตำแหน่งภาพ MRI ชุดตรวจสอบ ป้ายกำกับของผู้เชี่ยวชาญ และชื่อคลาส
# ----------------------------------------------------------------------------
# หาโฟลเดอร์ภาพชุดตรวจสอบ แล้วเก็บไว้ในตัวแปร VAL_DIR
#   - check_det_dataset("brain-tumor.yaml") : อ่านการตั้งค่าชุดข้อมูล brain-tumor ได้ dict ออกมา
#   - ["val"] : เปิดดูค่าที่กุญแจ "val" คือตำแหน่งโฟลเดอร์ภาพชุดตรวจสอบ
#   - Path(...) : แปลงข้อความตำแหน่งให้เป็นวัตถุ Path เพื่อใช้เครื่องมือจัดการตำแหน่งได้สะดวก
VAL_DIR = Path(check_det_dataset("brain-tumor.yaml")["val"])      # .../brain-tumor/images/val
# หาโฟลเดอร์ป้ายกำกับ (labels) ของภาพชุดตรวจสอบ ซึ่งเก็บคู่ขนานกับโฟลเดอร์ภาพ
#   - VAL_DIR.parent.parent : ถอยขึ้นไป 2 ระดับ จาก .../images/val เป็น .../brain-tumor
#   - / "labels" / VAL_DIR.name : ต่อชื่อโฟลเดอร์ "labels" และ "val" (VAL_DIR.name = ชื่อโฟลเดอร์สุดท้าย)
LBL_DIR = VAL_DIR.parent.parent / "labels" / VAL_DIR.name         # .../brain-tumor/labels/val
# สร้าง dict ชื่อคลาส {หมายเลข: ชื่อ} เช่น {0: 'negative', 1: 'positive'} ด้วย dict comprehension
#   - det.names.items() : ดึงคู่ (กุญแจ, ค่า) ทุกคู่จาก dict ชื่อคลาสของโมเดล
#   - {int(k): v for k, v in ...} : สร้าง dict ใหม่ โดยวนทีละคู่ แปลงกุญแจ k เป็นจำนวนเต็ม และใช้ค่า v เดิม
CLASS_NAMES = {int(k): v for k, v in det.names.items()}
# สร้างลิสต์ไฟล์ภาพชุดตรวจสอบทั้งหมด เรียงตามชื่อไฟล์ แล้วเก็บไว้ใน VAL_IMAGES
#   - VAL_DIR.iterdir() : รายการไฟล์ทั้งหมดในโฟลเดอร์ VAL_DIR
#   - p.suffix.lower() : นามสกุลไฟล์ (เช่น ".JPG") แปลงเป็นตัวพิมพ์เล็ก
#   - in {".jpg", ".jpeg", ".png"} : เลือกเฉพาะไฟล์ภาพ 3 นามสกุลนี้ ({ } ในที่นี้คือเซต = กลุ่มค่าที่ไม่ซ้ำ)
#   - sorted(...) : เรียงลำดับตามชื่อไฟล์ เพื่อให้ลำดับภาพคงที่ทุกครั้งที่รัน
VAL_IMAGES = sorted(p for p in VAL_DIR.iterdir() if p.suffix.lower() in {".jpg", ".jpeg", ".png"})


# ----------------------------------------------------------------------------
# ส่วนที่ 3: ฟังก์ชัน load_expert_boxes — อ่านกรอบของผู้เชี่ยวชาญในภาพหนึ่งภาพ
#   - img_path : ตำแหน่งไฟล์ภาพ MRI
#   - คืนค่า   : ลิสต์ของกรอบ แต่ละกรอบอยู่ในรูป (หมายเลขคลาส, x1, y1, x2, y2) หน่วยพิกเซล
#                (x1, y1) = มุมซ้ายบน และ (x2, y2) = มุมขวาล่างของกรอบ
#   ไฟล์ป้ายกำกับแบบ YOLO (.txt) มี 1 บรรทัดต่อ 1 กรอบ: "คลาส cx cy กว้าง สูง"
#   โดย cx, cy คือจุดกึ่งกลางกรอบ และทุกค่าเป็นสัดส่วน 0-1 ของความกว้าง/สูงของภาพ จึงต้องแปลงเป็นพิกเซล
# ----------------------------------------------------------------------------
def load_expert_boxes(img_path):
    """กรอบของผู้เชี่ยวชาญในภาพหนึ่งภาพ ในรูป [(cls, x1, y1, x2, y2)] หน่วยพิกเซล"""
    # เปิดไฟล์ภาพด้วย PILImage.open(...) แล้วอ่านขนาดภาพ .size ได้ (กว้าง, สูง) เก็บไว้ใน w และ h
    w, h = PILImage.open(img_path).size
    # สร้างตำแหน่งไฟล์ป้ายกำกับที่คู่กับภาพนี้
    #   - img_path.stem : ชื่อไฟล์ภาพโดยไม่มีนามสกุล (เช่น "00054_145")
    #   - + ".txt" : ต่อท้ายด้วยนามสกุล .txt แล้วนำไปต่อกับโฟลเดอร์ LBL_DIR
    lbl = LBL_DIR / (img_path.stem + ".txt")
    # สร้างลิสต์ว่างไว้เก็บกรอบของผู้เชี่ยวชาญ
    boxes = []
    # ทำต่อเมื่อมีไฟล์ป้ายกำกับอยู่จริงเท่านั้น (.exists() ตอบ True/False) — ภาพที่ไม่มีรอยโรคอาจไม่มีไฟล์นี้
    if lbl.exists():
        # วนซ้ำทีละบรรทัดในไฟล์ป้ายกำกับ
        #   - lbl.read_text() : อ่านเนื้อหาทั้งไฟล์เป็นข้อความ
        #   - .split("\n") : ตัดข้อความเป็นลิสต์ทีละบรรทัด ("\n" คือตัวขึ้นบรรทัดใหม่)
        for line in lbl.read_text().split("\n"):
            # ข้ามบรรทัดว่าง: line.strip() ตัดช่องว่างหัว-ท้าย หากเหลือข้อความ (ไม่ว่าง) จึงทำต่อ
            if line.strip():
                # แยกค่าในบรรทัดออกเป็น 5 ค่า
                #   - line.split() : ตัดข้อความตามช่องว่าง ได้ลิสต์ของข้อความ
                #   - [:5] : เอาเฉพาะ 5 ค่าแรก
                #   - map(float, ...) : แปลงทุกค่าเป็นตัวเลขทศนิยม
                #   - c, cx, cy, bw, bh = ... : แจกค่าทั้ง 5 ให้ตัวแปร 5 ตัวตามลำดับ
                #     (คลาส, จุดกึ่งกลาง x, จุดกึ่งกลาง y, ความกว้างกรอบ, ความสูงกรอบ)
                c, cx, cy, bw, bh = map(float, line.split()[:5])
                # แปลงจาก (กึ่งกลาง, กว้าง, สูง) แบบสัดส่วน เป็นมุมซ้ายบน/ขวาล่างหน่วยพิกเซล แล้วเพิ่มเข้าลิสต์
                #   - int(c) : หมายเลขคลาสเป็นจำนวนเต็ม
                #   - (cx - bw / 2) * w : x ซ้าย = กึ่งกลาง - ครึ่งความกว้าง แล้วคูณความกว้างภาพเป็นพิกเซล
                #   - (cy - bh / 2) * h : y บน
                #   - (cx + bw / 2) * w : x ขวา
                #   - (cy + bh / 2) * h : y ล่าง
                #   - ( ... ) คือทูเพิล (tuple) กลุ่มค่าเรียงลำดับที่แก้ไขไม่ได้
                boxes.append((int(c), (cx - bw / 2) * w, (cy - bh / 2) * h, (cx + bw / 2) * w, (cy + bh / 2) * h))
    # ส่งลิสต์กรอบของผู้เชี่ยวชาญกลับไปให้ผู้เรียกใช้ฟังก์ชัน
    return boxes


# ----------------------------------------------------------------------------
# ส่วนที่ 4: ฟังก์ชัน iou — คำนวณ IoU (Intersection over Union) ของกรอบ a และ b
#   - a, b : กรอบในรูป (x1, y1, x2, y2)
#   - IoU = พื้นที่ส่วนที่ซ้อนทับกัน ÷ พื้นที่รวมของทั้งสองกรอบ (ค่า 0 = ไม่ซ้อนกันเลย, 1 = ซ้อนกันพอดี)
# ----------------------------------------------------------------------------
def iou(a, b):
    # ความกว้างของส่วนที่ซ้อนทับ = ขอบขวาที่อยู่ซ้ายกว่า - ขอบซ้ายที่อยู่ขวากว่า
    #   - min(a[2], b[2]) : ขอบขวา (x2) ที่น้อยกว่า, max(a[0], b[0]) : ขอบซ้าย (x1) ที่มากกว่า
    #   - max(0.0, ...) : ถ้าผลติดลบ (กรอบไม่ซ้อนกัน) ให้เป็น 0
    ix = max(0.0, min(a[2], b[2]) - max(a[0], b[0]))
    # ความสูงของส่วนที่ซ้อนทับ คำนวณแบบเดียวกันในแนวตั้ง (a[1], a[3] คือ y1, y2)
    iy = max(0.0, min(a[3], b[3]) - max(a[1], b[1]))
    # พื้นที่ส่วนที่ซ้อนทับ (intersection) = กว้าง x สูง
    inter = ix * iy
    # พื้นที่รวม (union) = พื้นที่กรอบ a + พื้นที่กรอบ b - ส่วนที่ซ้อนทับ (ลบออกเพราะนับซ้ำ 2 ครั้ง)
    union = (a[2] - a[0]) * (a[3] - a[1]) + (b[2] - b[0]) * (b[3] - b[1]) - inter
    # ส่งค่า IoU กลับ: หาร inter ด้วย union หาก union มากกว่า 0 มิฉะนั้นคืนค่า 0.0 (ป้องกันการหารด้วยศูนย์)
    return inter / union if union > 0 else 0.0


# ----------------------------------------------------------------------------
# ส่วนที่ 5: ฟังก์ชัน judge — ตัดสินผลของกรอบโมเดลในภาพหนึ่งภาพ เทียบกับกรอบของผู้เชี่ยวชาญ
#   - preds     : กรอบของโมเดลในภาพนี้ แต่ละกรอบคือ (คลาส, ความเชื่อมั่น, x1, y1, x2, y2)
#   - experts   : กรอบของผู้เชี่ยวชาญ แต่ละกรอบคือ (คลาส, x1, y1, x2, y2)
#   - threshold : เกณฑ์ความเชื่อมั่น — เก็บเฉพาะกรอบของโมเดลที่มีความเชื่อมั่นไม่ต่ำกว่าค่านี้
#   - iou_thr=0.5 : เกณฑ์ IoU ขั้นต่ำที่ถือว่า "ตำแหน่งตรงกัน" (ค่าเริ่มต้น 0.5 หากไม่ระบุ)
#   คืนค่า 3 อย่าง: kept (กรอบที่ผ่านเกณฑ์), verdicts (ผลตัดสินของแต่ละกรอบ),
#                  found (กรอบของผู้เชี่ยวชาญแต่ละกรอบถูกตรวจพบหรือไม่)
#   ผลตัดสิน: "hit" = ถูกตำแหน่งและถูกคลาส (ผลบวกจริง true positive)
#            "wrong label" = ถูกตำแหน่งแต่ผิดคลาส
#            "false alarm" = แจ้งเตือนผิด ไม่ตรงกับกรอบใดของผู้เชี่ยวชาญ (ผลบวกลวง false positive)
# ----------------------------------------------------------------------------
def judge(preds, experts, threshold, iou_thr=0.5):
    """จำแนกกรอบของโมเดลแต่ละกรอบเป็น 'hit' / 'wrong label' / 'false alarm' และระบุว่ากรอบของผู้เชี่ยวชาญแต่ละกรอบตรวจพบหรือไม่"""
    # คัดเฉพาะกรอบที่ความเชื่อมั่นถึงเกณฑ์ แล้วเรียงจากความเชื่อมั่นมากไปน้อย
    #   - [p for p in preds if p[1] >= threshold] : list comprehension สร้างลิสต์ใหม่จากกรอบที่ p[1] (ความเชื่อมั่น) >= threshold
    #   - sorted(..., key=...) : เรียงลำดับตามค่าที่ key คำนวณได้
    #   - lambda p: -p[1] : ฟังก์ชันสั้นแบบไม่มีชื่อ (lambda) รับกรอบ p แล้วคืนค่าความเชื่อมั่นติดลบ
    #                       การเรียงค่าติดลบจากน้อยไปมาก = เรียงความเชื่อมั่นจากมากไปน้อย
    kept = sorted([p for p in preds if p[1] >= threshold], key=lambda p: -p[1])
    # สร้างลิสต์ False จำนวนเท่ากับกรอบของผู้เชี่ยวชาญ ([False] * n = ลิสต์ที่มี False n ตัว)
    #   ใช้บันทึกว่ากรอบของผู้เชี่ยวชาญแต่ละกรอบ "ถูกจับคู่กับกรอบโมเดลแล้วหรือยัง"
    found = [False] * len(experts)
    # สร้างลิสต์ว่างไว้เก็บผลตัดสินของกรอบโมเดลแต่ละกรอบ
    verdicts = []
    # วนซ้ำทีละกรอบของโมเดล (เริ่มจากกรอบที่มั่นใจที่สุด)
    #   - cls, conf, *box : แจกค่า ตัวแรกเป็นคลาส ตัวที่สองเป็นความเชื่อมั่น
    #                       และ *box รวบค่าที่เหลือทั้งหมด (x1, y1, x2, y2) เป็นลิสต์ชื่อ box
    for cls, conf, *box in kept:
        # ตั้งค่าเริ่มต้น: IoU ที่ดีที่สุดที่พบ = 0.0 และตำแหน่งกรอบผู้เชี่ยวชาญที่ดีที่สุด = -1 (ยังไม่พบ)
        best, best_j = 0.0, -1
        # วนซ้ำผ่านกรอบของผู้เชี่ยวชาญทุกกรอบ เพื่อหากรอบที่ตรงกับกรอบโมเดลนี้มากที่สุด
        #   - j : ลำดับที่ของกรอบผู้เชี่ยวชาญ, ecls : คลาส, *ebox : พิกัด (x1, y1, x2, y2)
        for j, (ecls, *ebox) in enumerate(experts):
            # เลือกเป็นคู่ที่ดีที่สุดใหม่ ถ้าครบทั้ง 3 เงื่อนไข
            #   - ecls == cls : คลาสตรงกัน
            #   - not found[j] : กรอบผู้เชี่ยวชาญนี้ยังไม่ถูกจับคู่ (กรอบหนึ่งนับเป็น hit ได้ครั้งเดียว)
            #   - iou(box, ebox) > best : ซ้อนทับมากกว่าคู่ที่ดีที่สุดเดิม
            if ecls == cls and not found[j] and iou(box, ebox) > best:
                # บันทึกค่า IoU ใหม่และลำดับที่ของกรอบผู้เชี่ยวชาญนี้ไว้
                best, best_j = iou(box, ebox), j
        # ถ้าคู่ที่ดีที่สุดมี IoU ถึงเกณฑ์ (>= iou_thr) ถือว่าเป็นการตรวจพบที่ถูกต้อง
        if best >= iou_thr:
            # ทำเครื่องหมายว่ากรอบผู้เชี่ยวชาญนี้ถูกตรวจพบแล้ว
            found[best_j] = True
            # บันทึกผลตัดสินของกรอบโมเดลนี้ว่า "hit"
            verdicts.append("hit")
        # elif (else if) = "มิฉะนั้น ถ้า..." ตรวจว่ากรอบนี้ซ้อนทับ (IoU >= iou_thr) กับกรอบผู้เชี่ยวชาญ "คลาสอื่น" หรือไม่
        #   - any(...) : ตอบ True ถ้ามีอย่างน้อย 1 กรอบที่เข้าเงื่อนไข
        #   - for ecls, *ebox in experts if ecls != cls : พิจารณาเฉพาะกรอบผู้เชี่ยวชาญที่คลาสไม่ตรงกับโมเดล
        elif any(iou(box, ebox) >= iou_thr for ecls, *ebox in experts if ecls != cls):
            # ถูกตำแหน่งแต่ผิดคลาส -> "wrong label"
            verdicts.append("wrong label")
        # else = กรณีอื่นทั้งหมด (ไม่ตรงกับกรอบใดของผู้เชี่ยวชาญ)
        else:
            # แจ้งเตือนผิด -> "false alarm"
            verdicts.append("false alarm")
    # ส่งค่ากลับ 3 อย่างพร้อมกัน (ผู้เรียกรับได้ด้วย a, b, c = judge(...))
    return kept, verdicts, found


# ----------------------------------------------------------------------------
# ส่วนที่ 6: ให้โมเดลทำนายภาพชุดตรวจสอบทุกภาพครั้งเดียว และอ่านกรอบของผู้เชี่ยวชาญ
# ----------------------------------------------------------------------------
# ให้โมเดลทำนายภาพ MRI ชุดตรวจสอบทุกภาพเพียงครั้งเดียว และเก็บทุกกรอบ (conf >= 0.001) ไว้สำหรับกรองภายหลัง
# สร้าง dict ว่าง 2 ตัวในบรรทัดเดียว
#   - PREDS   : เก็บกรอบของโมเดล โดยใช้ชื่อไฟล์ภาพเป็นกุญแจ
#   - EXPERTS : เก็บกรอบของผู้เชี่ยวชาญ โดยใช้ชื่อไฟล์ภาพเป็นกุญแจ
PREDS, EXPERTS = {}, {}
# วนซ้ำผ่านผลการทำนายทีละภาพ
#   - det.predict(...) : ให้โมเดลที่ฝึกแล้วทำนายภาพ
#       - source=str(VAL_DIR) : โฟลเดอร์ภาพชุดตรวจสอบ (str(...) แปลง Path เป็นข้อความ) ทำนายทุกภาพในโฟลเดอร์
#       - conf=0.001 : เก็บกรอบที่ความเชื่อมั่นตั้งแต่ 0.001 ขึ้นไป (ต่ำมาก = แทบทุกกรอบ) เพื่อกรองเองภายหลัง
#       - imgsz=416 : ขนาดภาพเท่ากับตอนฝึก
#       - stream=True : ส่งผลออกมาทีละภาพ ประหยัดหน่วยความจำ
#       - verbose=False : ไม่พิมพ์รายละเอียดของแต่ละภาพ
for r in det.predict(source=str(VAL_DIR), conf=0.001, imgsz=416, stream=True, verbose=False):
    # ดึงชื่อไฟล์ภาพ (ไม่รวมโฟลเดอร์) จากตำแหน่งไฟล์ r.path
    name = Path(r.path).name
    # เก็บกรอบทุกกรอบของภาพนี้ไว้ใน PREDS[name] ในรูป (คลาส, ความเชื่อมั่น, x1, y1, x2, y2)
    #   - r.boxes.xyxy.tolist() : พิกัดกรอบแบบ (x1, y1, x2, y2) หน่วยพิกเซล แปลงเป็นลิสต์ Python
    #   - r.boxes.cls.tolist()  : หมายเลขคลาสของแต่ละกรอบ
    #   - r.boxes.conf.tolist() : ความเชื่อมั่นของแต่ละกรอบ
    #   - zip(...) : จับคู่ค่าทั้ง 3 ของกรอบเดียวกัน ได้ b (พิกัด), c (คลาส), p (ความเชื่อมั่น)
    #   - (int(c), float(p), *map(float, b)) : สร้างทูเพิล โดย *map(...) กระจายพิกัด 4 ค่า (เป็นทศนิยม) ต่อท้าย
    PREDS[name] = [(int(c), float(p), *map(float, b))
                   for b, c, p in zip(r.boxes.xyxy.tolist(), r.boxes.cls.tolist(), r.boxes.conf.tolist())]
# วนซ้ำผ่านไฟล์ภาพชุดตรวจสอบทุกไฟล์
for p in VAL_IMAGES:
    # อ่านกรอบของผู้เชี่ยวชาญของภาพนี้ แล้วเก็บไว้ใน EXPERTS โดยใช้ชื่อไฟล์ (p.name) เป็นกุญแจ
    EXPERTS[p.name] = load_expert_boxes(p)
# พิมพ์สรุปจำนวนกรอบ (f-string 2 ชิ้นที่เขียนติดกันจะถูกต่อเป็นข้อความเดียว)
#   - sum(len(v) for v in PREDS.values()) : รวมจำนวนกรอบของโมเดลในทุกภาพ
#   - len(PREDS) : จำนวนภาพที่ทำนาย
#   - sum(len(v) for v in EXPERTS.values()) : รวมจำนวนกรอบของผู้เชี่ยวชาญในทุกภาพ
print(f"โมเดลเสนอกรอบที่เป็นไปได้ {sum(len(v) for v in PREDS.values())} กรอบ บนภาพ MRI ชุดตรวจสอบ {len(PREDS)} ภาพ; "
      f"ผู้เชี่ยวชาญวาดกรอบไว้ {sum(len(v) for v in EXPERTS.values())} กรอบ")

# กำหนดสีของกรอบตามผลตัดสิน (dict): hit = เขียว (lime), wrong label = ส้ม, false alarm = แดง
COLORS = {"hit": "lime", "wrong label": "orange", "false alarm": "red"}


# ----------------------------------------------------------------------------
# ส่วนที่ 7: ฟังก์ชัน show_detections — แสดงภาพ MRI พร้อมกรอบของผู้เชี่ยวชาญและของโมเดล
#   - threshold=0.25 : เกณฑ์ความเชื่อมั่น แสดงเฉพาะกรอบของโมเดลที่ความเชื่อมั่น >= ค่านี้
#   - start=0 : เริ่มแสดงจากภาพลำดับที่เท่าใดใน VAL_IMAGES (0 = ภาพแรก)
#   - n=8 : จำนวนภาพที่จะแสดง
#   - only_errors=False : หากเป็น True จะข้ามภาพที่โมเดลทำถูกทั้งหมด แสดงเฉพาะภาพที่มีความคลาดเคลื่อน
#   ค่าหลังเครื่องหมาย = ในวงเล็บคือ "ค่าเริ่มต้น" ซึ่งใช้เมื่อผู้เรียกไม่ได้ระบุพารามิเตอร์นั้น
# ----------------------------------------------------------------------------
def show_detections(threshold=0.25, start=0, n=8, only_errors=False):
    """แสดงภาพ MRI ชุดตรวจสอบ n ภาพ พร้อมกรอบของผู้เชี่ยวชาญ (เส้นประ) และกรอบของโมเดลที่มีความเชื่อมั่น >= threshold"""
    # หมายเหตุ: ข้อความบนภาพใช้ภาษาอังกฤษ เนื่องจากฟอนต์เริ่มต้นของ matplotlib บน Colab ไม่รองรับภาษาไทย
    # สร้างลิสต์ว่างไว้เก็บภาพที่เลือกมาแสดง
    chosen = []
    # วนซ้ำผ่านภาพตั้งแต่ลำดับที่ start จนถึงภาพสุดท้าย ([start:] = ตัดช่วงตั้งแต่ start ถึงท้ายลิสต์)
    for p in VAL_IMAGES[start:]:
        # ตัดสินผลของภาพนี้ด้วยฟังก์ชัน judge แล้วรับค่ากลับ 3 ตัว
        #   - PREDS.get(p.name, []) : กรอบของโมเดลในภาพนี้ หากไม่มีข้อมูลให้ใช้ลิสต์ว่าง [] แทน
        #   - EXPERTS[p.name] : กรอบของผู้เชี่ยวชาญในภาพนี้
        #   - threshold : เกณฑ์ความเชื่อมั่นที่ผู้เรียนกำหนด
        kept, verdicts, found = judge(PREDS.get(p.name, []), EXPERTS[p.name], threshold)
        # ถ้าต้องการเฉพาะภาพที่ผิดพลาด และภาพนี้ถูกต้องทั้งหมด ให้ข้ามไป
        #   - all(v == "hit" for v in verdicts) : กรอบของโมเดลทุกกรอบเป็น hit
        #   - all(found) : กรอบของผู้เชี่ยวชาญทุกกรอบถูกตรวจพบ
        if only_errors and all(v == "hit" for v in verdicts) and all(found):
            # continue = ข้ามคำสั่งที่เหลือในรอบนี้ แล้วไปภาพถัดไป
            continue
        # เก็บข้อมูลของภาพนี้ (ไฟล์, กรอบที่ผ่านเกณฑ์, ผลตัดสิน, การตรวจพบ) เป็นทูเพิลต่อท้ายลิสต์ chosen
        chosen.append((p, kept, verdicts, found))
        # ถ้าเก็บครบ n ภาพแล้ว
        if len(chosen) == n:
            # break = ออกจากการวนซ้ำทันที
            break
    # ถ้าไม่มีภาพที่เลือกได้เลย (not chosen = ลิสต์ว่าง)
    if not chosen:
        # พิมพ์ข้อความแนะนำ แล้ว return เพื่อจบฟังก์ชันทันทีโดยไม่วาดภาพ (2 คำสั่งคั่นด้วย ;)
        print("ไม่มีภาพที่จะแสดง (โปรดลดค่า start หรือใช้ only_errors=False)"); return
    # กำหนดให้แสดง 4 ภาพต่อแถว
    cols = 4
    # คำนวณจำนวนแถวที่ต้องใช้ แบบปัดขึ้น: (จำนวนภาพ + 3) // 4  (// คือการหารแบบปัดเศษทิ้ง)
    rows = (len(chosen) + cols - 1) // cols
    # สร้างพื้นที่วาดภาพแบ่งเป็นตาราง rows แถว x cols คอลัมน์
    #   - figsize=(4 * cols, 4.3 * rows) : กว้าง 4 นิ้วต่อคอลัมน์ สูง 4.3 นิ้วต่อแถว
    #   - squeeze=False : ให้ axes เป็นตาราง 2 มิติเสมอ
    fig, axes = plt.subplots(rows, cols, figsize=(4 * cols, 4.3 * rows), squeeze=False)
    # วนซ้ำผ่านทุกช่องย่อย (axes.flat = เรียงช่องย่อยทุกช่องเป็นแถวเดียว)
    for ax in axes.flat:
        # ซ่อนแกนพิกัดของทุกช่อง (ช่องที่ไม่มีภาพจะกลายเป็นพื้นที่ว่าง)
        ax.axis("off")
    # วนซ้ำจับคู่ช่องย่อยกับภาพที่เลือกไว้ แล้วแยกข้อมูลของแต่ละภาพออกเป็น p, kept, verdicts, found
    for ax, (p, kept, verdicts, found) in zip(axes.flat, chosen):
        # เปิดไฟล์ภาพ แปลงเป็นภาพสี RGB (.convert("RGB")) แล้ววาดลงในช่องย่อย
        ax.imshow(PILImage.open(p).convert("RGB"))
        # วาดกรอบของผู้เชี่ยวชาญทีละกรอบ คู่กับสถานะว่าตรวจพบหรือไม่ (ok = True/False)
        for (ecls, x1, y1, x2, y2), ok in zip(EXPERTS[p.name], found):
            # เพิ่มกรอบสี่เหลี่ยมลงในภาพ (ax.add_patch) โดยสร้างกรอบด้วย patches.Rectangle(...)
            #   - (x1, y1) : มุมเริ่มต้นของกรอบ
            #   - x2 - x1 : ความกว้างกรอบ, y2 - y1 : ความสูงกรอบ
            #   - fill=False : ไม่ระบายสีภายในกรอบ
            #   - ls="--" : (linestyle) เส้นประ — ใช้แทนกรอบของผู้เชี่ยวชาญ
            #   - lw=2 : (linewidth) ความหนาเส้น 2
            #   - ec=... : (edgecolor) สีเส้นขอบ: ขาวถ้าตรวจพบ เหลืองถ้าตรวจไม่พบ
            ax.add_patch(patches.Rectangle((x1, y1), x2 - x1, y2 - y1, fill=False, ls="--", lw=2,
                                           ec="white" if ok else "yellow"))
            # เขียนข้อความใต้กรอบ (ax.text)
            #   - x1, y2 + 12 : ตำแหน่งข้อความ ชิดซ้ายของกรอบ และต่ำกว่าขอบล่าง 12 พิกเซล
            #                  (ในภาพ ค่า y ยิ่งมากยิ่งอยู่ด้านล่าง)
            #   - ข้อความ : "expert: ชื่อคลาส" และต่อท้าย "← MISSED" หากโมเดลตรวจไม่พบ (ผลลบลวง false negative)
            #   - color : สีข้อความ ขาว/เหลือง ตามสถานะ
            #   - fontsize=8 : ขนาดตัวอักษร, weight="bold" : ตัวหนา
            ax.text(x1, y2 + 12, f"expert: {CLASS_NAMES[ecls]}" + ("" if ok else "  ← MISSED"),
                    color="white" if ok else "yellow", fontsize=8, weight="bold")
        # วาดกรอบของโมเดลที่ผ่านเกณฑ์ทีละกรอบ คู่กับผลตัดสิน v
        for (cls, conf, x1, y1, x2, y2), v in zip(kept, verdicts):
            # เพิ่มกรอบเส้นทึบ (ไม่กำหนด ls จึงเป็นเส้นทึบ) ความหนา 2 สีตามผลตัดสินจาก COLORS[v]
            ax.add_patch(patches.Rectangle((x1, y1), x2 - x1, y2 - y1, fill=False, lw=2, ec=COLORS[v]))
            # เขียนข้อความเหนือกรอบ (y1 - 4) เป็น "ชื่อคลาส ความเชื่อมั่น(ทศนิยม 2 ตำแหน่ง) (ผลตัดสิน)" สีเดียวกับกรอบ
            ax.text(x1, y1 - 4, f"{CLASS_NAMES[cls]} {conf:.2f} ({v})", color=COLORS[v], fontsize=8, weight="bold")
        # นับจำนวนกรอบผู้เชี่ยวชาญที่ตรวจไม่พบ (.count(False) = นับจำนวน False ในลิสต์)
        n_miss = found.count(False)
        # นับจำนวนกรอบของโมเดลที่ไม่ใช่ hit (แจ้งเตือนผิด + ผิดคลาส); True นับเป็น 1 เมื่อนำมา sum
        n_fa = sum(v != "hit" for v in verdicts)
        # ตั้งชื่อช่องย่อย: ชื่อไฟล์ ขึ้นบรรทัดใหม่ (\n) แล้วตามด้วยจำนวนที่ตรวจไม่พบและที่ผิดพลาด ขนาดอักษร 9
        ax.set_title(f"{p.name}\nmissed: {n_miss}  false alarms/wrong label: {n_fa}", fontsize=9)
    # ตั้งชื่อหลักของภาพรวมทั้งหมด (fig.suptitle) อธิบายความหมายของเส้นและสี ขนาดอักษร 10
    fig.suptitle(f"threshold = {threshold}   |   dashed = expert box (yellow = missed)   |   "
                 "solid = model: green hit, orange right place/wrong label, red false alarm", fontsize=10)
    # จัดระยะช่องย่อยไม่ให้ทับกัน แล้วแสดงภาพออกมาใต้เซลล์
    plt.tight_layout(); plt.show()


# ----------------------------------------------------------------------------
# ส่วนที่ 8: ฟังก์ชัน count_errors — นับผลการตรวจจับบนภาพชุดตรวจสอบทั้งหมด แยกตาม threshold
#   - thresholds=(0.25,) : เกณฑ์ความเชื่อมั่นหนึ่งค่าหรือหลายค่า (ค่าเริ่มต้นคือทูเพิลที่มีค่าเดียว 0.25)
#   - คืนค่า : ตาราง pandas DataFrame 1 แถวต่อ 1 threshold แสดงจำนวน hit / missed / false alarm
#             ความไว (sensitivity) และค่าพยากรณ์บวก (PPV หรือ precision)
# ----------------------------------------------------------------------------
def count_errors(thresholds=(0.25,)):
    """จำนวนการตรวจพบที่ถูกต้อง การตรวจไม่พบ และการแจ้งเตือนผิด ในภาพ MRI ชุดตรวจสอบทั้ง 223 ภาพ แยกตาม threshold"""
    # ถ้าผู้เรียกส่งตัวเลขเดี่ยวมา (ไม่ใช่ลิสต์)
    #   - isinstance(x, (int, float)) : ตรวจว่า x เป็นจำนวนเต็มหรือทศนิยมหรือไม่
    if isinstance(thresholds, (int, float)):
        # ห่อให้เป็นลิสต์ที่มีค่าเดียว เพื่อให้วนซ้ำด้านล่างได้เหมือนกัน
        thresholds = [thresholds]
    # นับจำนวนกรอบของผู้เชี่ยวชาญทั้งหมดในทุกภาพ (ตัวหารของความไว)
    n_expert = sum(len(v) for v in EXPERTS.values())
    # สร้างลิสต์ว่างไว้เก็บผลแต่ละ threshold (1 แถวของตาราง)
    rows = []
    # วนซ้ำทีละ threshold
    for t in thresholds:
        # ตั้งตัวนับ 3 ตัวให้เป็น 0 พร้อมกัน (hits = จำนวน hit, fa = false alarm, wrong = wrong label)
        hits = fa = wrong = 0
        # วนซ้ำทุกภาพ: name = ชื่อไฟล์, experts = กรอบของผู้เชี่ยวชาญในภาพนั้น (.items() ให้คู่กุญแจ-ค่า)
        for name, experts in EXPERTS.items():
            # ตัดสินผลของภาพนี้ที่ threshold t; ใช้เครื่องหมาย _ รับค่าที่ไม่ต้องการ (kept และ found) แล้วทิ้งไป
            _, verdicts, _ = judge(PREDS.get(name, []), experts, t)
            # สะสมจำนวนแต่ละผลตัดสิน (+= คือบวกเพิ่มเข้าไปในค่าเดิม; .count(...) นับจำนวนข้อความนั้นในลิสต์)
            hits += verdicts.count("hit"); wrong += verdicts.count("wrong label"); fa += verdicts.count("false alarm")
        # จำนวนกรอบที่โมเดลรายงานทั้งหมดที่ threshold นี้ (ตัวหารของ PPV)
        n_pred = hits + wrong + fa
        # เพิ่มผลของ threshold นี้เป็น dict 1 แถว (กุญแจ = ชื่อคอลัมน์) ต่อท้ายลิสต์ rows
        #   - "threshold" : ค่าเกณฑ์
        #   - กรอบของผู้เชี่ยวชาญ : n_expert
        #   - ตรวจพบถูกต้อง : hits (ผลบวกจริง)
        #   - ตรวจไม่พบ : n_expert - hits (ผลลบลวง)
        #   - แจ้งเตือนผิด : fa (ผลบวกลวง)
        #   - ระบุคลาสผิด : wrong
        #   - ความไว (sensitivity) = hits ÷ n_expert ปัดทศนิยม 3 ตำแหน่งด้วย round(..., 3)
        #   - PPV (precision) = hits ÷ n_pred หาก n_pred ไม่เป็น 0 มิฉะนั้นใช้ float("nan") = ค่าว่างทางตัวเลข (Not a Number)
        rows.append({"threshold": t, "กรอบของผู้เชี่ยวชาญ (expert boxes)": n_expert, "ตรวจพบถูกต้อง (hits)": hits,
                     "ตรวจไม่พบ (missed)": n_expert - hits, "แจ้งเตือนผิด (false alarms)": fa,
                     "ระบุคลาสผิด (wrong label)": wrong,
                     "ความไว (sensitivity)": round(hits / n_expert, 3),
                     "PPV (precision)": round(hits / n_pred, 3) if n_pred else float("nan")})
    # แปลงลิสต์ของ dict เป็นตาราง (pd.DataFrame) แล้วใช้คอลัมน์ "threshold" เป็นชื่อแถว (.set_index) และส่งกลับ
    return pd.DataFrame(rows).set_index("threshold")


# พิมพ์ข้อความยืนยันว่าเครื่องมือทั้งสองพร้อมใช้งานแล้ว
print("เครื่องมือพร้อมใช้งาน: show_detections(threshold, start=0) และ count_errors([thresholds])")

In [ ]:
# ⭐ Exercise 3 — แบบฝึกหัดที่ 3: ตรวจพินิจผลการตรวจจับของโมเดลที่เกณฑ์ความเชื่อมั่นหนึ่งค่า
# หน้าที่ของเซลล์: แสดงภาพ MRI ชุดตรวจสอบ 8 ภาพ กรอบเส้นประแสดงตำแหน่งรอยโรคที่ผู้เชี่ยวชาญกำหนด
#   กรอบเส้นทึบแสดงกรอบที่โมเดลตรวจพบซึ่งมีความเชื่อมั่น >= THRESHOLD
# สิ่งที่ต้องแก้ไข: กำหนด THRESHOLD เป็นตัวเลขระหว่าง 0 ถึง 1 (โมเดลจะรายงานเฉพาะกรอบที่มีความเชื่อมั่นไม่ต่ำกว่าค่านี้)
#   เมื่อรันแล้ว ให้ทดลองค่าต่ำ (0.1) และค่าสูง (0.7) เพื่อเปรียบเทียบ
#   และเปลี่ยนค่า START (เช่น 8, 16, 40, 100) เพื่อแสดงภาพ MRI ชุดอื่น
# คำแนะนำ: THRESHOLD = 0.25 (ค่าเริ่มต้นของ Ultralytics)
# ============================================================================
# 🎯 วัตถุประสงค์ของเซลล์นี้
#   ใช้เครื่องมือจากเซลล์ 🔧 ด้านบน เพื่อ "ดูด้วยตา" ว่าโมเดลตรวจจับรอยโรคถูกหรือผิดอย่างไร
#   ที่เกณฑ์ตัดสิน (threshold) ความเชื่อมั่นหนึ่งค่า
#   ผลลัพธ์ที่ผู้เรียนจะเห็น:
#   1) ภาพ MRI 8 ภาพ: กรอบเส้นประ = ผู้เชี่ยวชาญ (สีเหลือง = โมเดลตรวจไม่พบ)
#      กรอบเส้นทึบ = โมเดล (เขียว = ถูกต้อง, ส้ม = ถูกตำแหน่งแต่ผิดคลาส, แดง = แจ้งเตือนผิด)
#   2) ตารางสรุปบนภาพชุดตรวจสอบทั้ง 223 ภาพ: จำนวนที่ตรวจพบ/ตรวจไม่พบ/แจ้งเตือนผิด
#      ความไว (sensitivity) และค่าพยากรณ์บวก (PPV)
#   ผู้เรียนต้องกรอกค่า THRESHOLD ในตำแหน่ง ... ก่อนรันเซลล์
# ============================================================================

# กำหนดเกณฑ์ความเชื่อมั่น (threshold) เก็บไว้ในตัวแปร THRESHOLD
#   - ... คือช่องว่างที่ผู้เรียนต้องแทนที่ด้วยตัวเลขระหว่าง 0 ถึง 1 (ดูคำแนะนำด้านบน)
#   - ค่าต่ำ = โมเดลรายงานกรอบมากขึ้น, ค่าสูง = รายงานเฉพาะกรอบที่มั่นใจมาก
THRESHOLD = ...
# กำหนดลำดับภาพแรกที่จะแสดง (0 = ภาพแรกของชุดตรวจสอบ) เปลี่ยนเป็น 8, 16, ... เพื่อดูภาพชุดอื่น
START = 0

# เรียกฟังก์ชัน show_detections (สร้างไว้ในเซลล์ 🔧) เพื่อแสดงภาพ MRI 8 ภาพพร้อมกรอบ
#   - THRESHOLD : เกณฑ์ความเชื่อมั่นที่ใช้กรองกรอบของโมเดล
#   - start=START : เริ่มแสดงจากภาพลำดับที่ START
show_detections(THRESHOLD, start=START)
# เรียกฟังก์ชัน count_errors เพื่อสร้างตารางสรุปจำนวนความคลาดเคลื่อนบนภาพทั้ง 223 ภาพ ที่ THRESHOLD นี้
#   - เนื่องจากเป็นบรรทัดสุดท้ายของเซลล์ Jupyter/Colab จะแสดงค่าที่ได้ (ตาราง) ออกมาให้อัตโนมัติ
#     โดยไม่ต้องใช้ print หรือ display
count_errors(THRESHOLD)

In [ ]:
# @title ✅ เฉลย — หากไม่สามารถทำแบบฝึกหัดได้ ให้รันเซลล์นี้ (ดับเบิลคลิกเพื่อแสดงโค้ด) { display-mode: "form" }
# ============================================================================
# 🎯 วัตถุประสงค์ของเซลล์นี้ (เฉลยแบบฝึกหัดที่ 3)
#   แสดงภาพ MRI ชุดตรวจสอบ 8 ภาพ พร้อมกรอบของผู้เชี่ยวชาญ (เส้นประ) และกรอบของโมเดล (เส้นทึบ)
#   ที่เกณฑ์ความเชื่อมั่น 0.25 (ค่าเริ่มต้นของ Ultralytics)
#   แล้วแสดงตารางสรุปจำนวนการตรวจพบ ตรวจไม่พบ แจ้งเตือนผิด ความไว (sensitivity) และ PPV บนภาพทั้ง 223 ภาพ
# ============================================================================

# กำหนดเกณฑ์ความเชื่อมั่น = 0.25 (โมเดลรายงานเฉพาะกรอบที่มีความเชื่อมั่นไม่ต่ำกว่า 0.25)
THRESHOLD = 0.25
# เริ่มแสดงจากภาพแรกของชุดตรวจสอบ (ลำดับที่ 0)
START = 0

# แสดงภาพ MRI 8 ภาพพร้อมกรอบ ด้วยฟังก์ชัน show_detections จากเซลล์ 🔧
#   - THRESHOLD : เกณฑ์ความเชื่อมั่น, start=START : ลำดับภาพแรกที่จะแสดง
show_detections(THRESHOLD, start=START)
# สร้างตารางสรุปความคลาดเคลื่อนที่ THRESHOLD นี้ (บรรทัดสุดท้ายของเซลล์ จึงแสดงตารางออกมาอัตโนมัติ)
count_errors(THRESHOLD)

💡 **คำอธิบายผลลัพธ์** (ข้อความบนภาพเป็นภาษาอังกฤษ)
- **กรอบเส้นประสีขาว** คือรอยโรคที่ผู้เชี่ยวชาญกำหนดและโมเดลตรวจพบ ส่วน **กรอบเส้นประสีเหลือง ← MISSED** คือรอยโรคที่โมเดล *ตรวจไม่พบ* (ผลลบลวง (false negative))
- **กรอบสีเขียว (hit)** คือการตรวจพบที่ถูกต้อง (ผลบวกจริง (true positive)) **กรอบสีแดง (false alarm)** คือการแจ้งเตือนผิด (ผลบวกลวง (false positive)) ซึ่งโมเดลระบุรอยโรคในตำแหน่งที่ผู้เชี่ยวชาญไม่ได้กำหนด **กรอบสีส้ม (wrong label)** คือตำแหน่งถูกต้องแต่ระบุคลาสผิด (สลับระหว่าง `negative` และ `positive`)
- ตัวเลขหลังชื่อคลาสคือค่าความเชื่อมั่นของโมเดล
- ในตาราง **ความไว (sensitivity)** คือสัดส่วนของรอยโรคที่ผู้เชี่ยวชาญกำหนดซึ่งโมเดลตรวจพบ และ **PPV** (ค่าพยากรณ์บวก (positive predictive value / precision)) คือสัดส่วนของกรอบที่โมเดลรายงานซึ่งถูกต้อง

✍️ **คำถามที่ 2** ให้ตรวจพินิจภาพ MRI อย่างน้อย 16 ภาพ (โดยเปลี่ยนค่า `START` หรือรัน `show_detections(THRESHOLD, only_errors=True)` ในเซลล์ใหม่เพื่อแสดงเฉพาะภาพที่มีความคลาดเคลื่อน) จงอธิบาย **รอยโรคที่โมเดลตรวจไม่พบหนึ่งตำแหน่ง** และ **การแจ้งเตือนผิดหรือการระบุคลาสผิดหนึ่งตำแหน่ง** โดยระบุชื่อภาพและลักษณะที่พบ (เช่น ขนาดเล็ก ความเข้มต่ำ หรืออยู่ใกล้ขอบกะโหลกศีรษะ) พร้อมทั้งอธิบายผลกระทบทางคลินิกที่อาจเกิดขึ้นในแต่ละกรณี

✍️ **คำตอบ:** 

In [ ]:
# ⭐ Exercise 4 — แบบฝึกหัดที่ 4: เลือกเกณฑ์ตัดสิน (threshold) สำหรับการใช้งานทางคลินิก
# หน้าที่ของเซลล์: เปรียบเทียบจำนวนการตรวจไม่พบและการแจ้งเตือนผิดที่ threshold หลายค่า บนภาพ MRI ชุดตรวจสอบทั้ง 223 ภาพ
#   แล้วแสดงเฉพาะภาพที่โมเดลมีความคลาดเคลื่อน ณ threshold ที่ผู้เรียนเลือก
# สิ่งที่ต้องแก้ไข: กำหนด MY_THRESHOLD เป็นค่าที่เหมาะสมสำหรับ "เครื่องมือคัดกรอง" ซึ่งส่งภาพ MRI ให้รังสีแพทย์ทบทวน
#   (การตรวจไม่พบเนื้องอกมีผลเสียร้ายแรงกว่าการให้แพทย์ทบทวนภาพเพิ่มเติม)
# คำแนะนำ: พิจารณาจากตาราง และเลือก threshold ที่ให้ความไวสูงโดยมีจำนวนการแจ้งเตือนผิดที่ยอมรับได้ เช่น MY_THRESHOLD = 0.1
# ============================================================================
# 🎯 วัตถุประสงค์ของเซลล์นี้
#   ฝึกเลือก "จุดทำงาน (operating point)" ของโมเดลให้เหมาะกับการใช้งานทางคลินิก
#   - threshold ต่ำ -> ตรวจไม่พบน้อยลง (ความไวสูงขึ้น) แต่แจ้งเตือนผิดมากขึ้น (PPV ลดลง)
#   - threshold สูง -> ผลในทิศทางตรงข้าม
#   ตัวโมเดลไม่ได้เปลี่ยน สิ่งที่เปลี่ยนคือเกณฑ์ที่เลือกใช้
#   ผลลัพธ์ที่ผู้เรียนจะเห็น:
#   1) ตารางเปรียบเทียบ 5 threshold (0.05, 0.1, 0.25, 0.5, 0.75)
#   2) ภาพ MRI เฉพาะภาพที่โมเดลผิดพลาด (ตรวจไม่พบ / แจ้งเตือนผิด / ผิดคลาส) ณ threshold ที่ผู้เรียนเลือก
#   ผู้เรียนต้องกรอกค่า MY_THRESHOLD ในตำแหน่ง ... ก่อนรันเซลล์
# ============================================================================

# สร้างและแสดงตารางเปรียบเทียบผลที่ threshold 5 ค่า
#   - count_errors([...]) : ฟังก์ชันจากเซลล์ 🔧 รับลิสต์ของ threshold แล้วคืนตาราง 1 แถวต่อ 1 threshold
#   - [0.05, 0.1, 0.25, 0.5, 0.75] : ลิสต์ threshold ที่ต้องการเปรียบเทียบ
#   - display(...) : แสดงตารางออกมาทันที (จำเป็นต้องใช้ เพราะบรรทัดนี้ไม่ใช่บรรทัดสุดท้ายของเซลล์)
display(count_errors([0.05, 0.1, 0.25, 0.5, 0.75]))

# กำหนด threshold ที่ผู้เรียนเลือกสำหรับเครื่องมือคัดกรอง เก็บไว้ในตัวแปร MY_THRESHOLD
#   - ... คือช่องว่างที่ผู้เรียนต้องแทนที่ด้วยตัวเลขระหว่าง 0 ถึง 1 โดยพิจารณาจากตารางด้านบน (ดูคำแนะนำ)
MY_THRESHOLD = ...

# แสดงภาพ MRI ที่โมเดลมีความคลาดเคลื่อน ณ threshold ที่เลือก ด้วยฟังก์ชัน show_detections
#   - MY_THRESHOLD : เกณฑ์ความเชื่อมั่นที่ผู้เรียนเลือก
#   - only_errors=True : ข้ามภาพที่โมเดลทำถูกทั้งหมด แสดงเฉพาะภาพที่มีการตรวจไม่พบ แจ้งเตือนผิด หรือผิดคลาส
show_detections(MY_THRESHOLD, only_errors=True)

In [ ]:
# @title ✅ เฉลย — หากไม่สามารถทำแบบฝึกหัดได้ ให้รันเซลล์นี้ (ดับเบิลคลิกเพื่อแสดงโค้ด) { display-mode: "form" }
# ============================================================================
# 🎯 วัตถุประสงค์ของเซลล์นี้ (เฉลยแบบฝึกหัดที่ 4)
#   เปรียบเทียบผลการตรวจจับที่ threshold 5 ค่า แล้วเลือก threshold = 0.1 สำหรับ "เครื่องมือคัดกรอง"
#   เหตุผล: ในการคัดกรอง การตรวจไม่พบเนื้องอก (ผลลบลวง) มีผลเสียร้ายแรงกว่าการแจ้งเตือนผิด
#   ซึ่งรังสีแพทย์ทบทวนภาพเพิ่มเติมได้ จึงเลือก threshold ต่ำเพื่อให้ความไว (sensitivity) สูง
#   ผลลัพธ์ : ตารางเปรียบเทียบ และภาพ MRI เฉพาะภาพที่โมเดลยังผิดพลาดที่ threshold 0.1
# ============================================================================

# สร้างและแสดงตารางเปรียบเทียบผลที่ threshold 5 ค่า ด้วย count_errors (จากเซลล์ 🔧) และ display(...)
#   - [0.05, 0.1, 0.25, 0.5, 0.75] : ลิสต์ threshold ที่ต้องการเปรียบเทียบ
display(count_errors([0.05, 0.1, 0.25, 0.5, 0.75]))

# เลือก threshold = 0.1 : ความไวสูง โดยจำนวนการแจ้งเตือนผิดยังอยู่ในระดับที่รังสีแพทย์ทบทวนได้
MY_THRESHOLD = 0.1

# แสดงเฉพาะภาพ MRI ที่ยังมีความคลาดเคลื่อน ณ threshold 0.1
#   - MY_THRESHOLD : เกณฑ์ความเชื่อมั่นที่เลือก
#   - only_errors=True : ข้ามภาพที่โมเดลทำถูกทั้งหมด
show_detections(MY_THRESHOLD, only_errors=True)

💡 **คำอธิบายผลลัพธ์**
เมื่อลด threshold โมเดลจะรายงานกรอบมากขึ้น ส่งผลให้ **ตรวจไม่พบรอยโรคน้อยลง** (ความไวสูงขึ้น) แต่ **แจ้งเตือนผิดมากขึ้น** (PPV ลดลง) และเมื่อเพิ่ม threshold จะให้ผลในทิศทางตรงข้าม ตัวโมเดลไม่ได้เปลี่ยนแปลง สิ่งที่เปลี่ยนคือจุดทำงาน (operating point) ที่เลือก ค่า mAP สรุปประสิทธิภาพของโมเดลครอบคลุม *ทุก* threshold แต่ในการใช้งานจริง ทีมผู้ให้บริการทางคลินิกต้องเลือก threshold หนึ่งค่าที่สอดคล้องกับวัตถุประสงค์ของการใช้งาน (การคัดกรองหรือการยืนยันการวินิจฉัย)

✍️ **คำถามที่ 3** โปรดระบุค่า `MY_THRESHOLD` ที่เลือก พร้อมค่าความไวและจำนวนการแจ้งเตือนผิดที่ได้ หากเครื่องมือนี้ออกรายงานผลขั้นสุดท้ายโดยอัตโนมัติ *โดยไม่มี* รังสีแพทย์ตรวจทาน ผู้เรียนพิจารณาว่าควรเลือก threshold ต่างไปจากเดิมหรือไม่ เพราะเหตุใด

✍️ **คำตอบ:** 

✍️ **คำถามที่ 4** สมมติว่านำโมเดลนี้ไปใช้ตรวจหา **polyp ในวิดีโอการส่องกล้องลำไส้ใหญ่ (colonoscopy)** ซึ่งทุกเฟรมที่ตรวจไม่พบอาจหมายถึงรอยโรคที่ถูกมองข้าม จากเนื้อหาในปฏิบัติการนี้ (การประมวลผลวิดีโอทีละเฟรม ความคลาดเคลื่อนที่วิเคราะห์ และการแลกเปลี่ยนในการเลือก threshold) โปรดระบุความเสี่ยงหนึ่งข้อ และมาตรการป้องกันหนึ่งข้อที่ควรกำหนดก่อนนำไปใช้กับผู้ป่วย

✍️ **คำตอบ:** 

## 🏆 โจทย์ท้าทายบนตารางอันดับ (Leaderboard) — ไม่บังคับ สำหรับผู้เรียนที่เขียนโปรแกรมได้

> **ส่วนนี้ไม่บังคับ และไม่เป็นส่วนหนึ่งของชิ้นงานที่ต้องส่ง** ผู้เรียนที่ไม่ประสงค์จะทำสามารถข้ามส่วนนี้ได้ โดยชิ้นงานยังคงครบถ้วน

ให้ส่งกรอบที่โมเดลทำนายทั้งหมดบน **ภาพชุด val 223 ภาพ** ในรูปแบบลิสต์ JSON ของ `{image, class, conf, bbox:[x1,y1,x2,y2]}` (พิกัดหน่วยพิกเซล ไฟล์ `lab05_submission.json`) **ตัวชี้วัด: mAP@0.5** คำนวณบนเซิร์ฟเวอร์ (หลักการเดียวกับ `metrics.box.map50`) ควรใช้ **เกณฑ์ความเชื่อมั่นต่ำมาก** (baseline ใช้ `conf=0.001`) เนื่องจากการคำนวณ mAP ต้องใช้รายการกรอบทั้งหมดที่เรียงตามความเชื่อมั่น มิใช่เฉพาะกรอบที่มีความเชื่อมั่นสูง

วิธีส่งผล: ระบุ `NAME` และ `JOIN_CODE` ในเซลล์ ⚙️ การตั้งค่า (ผู้สอนประกาศรหัสเข้าร่วมในการเรียนการสอนผ่าน Zoom) แล้วรันเซลล์ดังกล่าวอีกครั้ง โค้ด baseline ด้านล่างใช้งานได้ทันทีกับโมเดล `det` จากแบบฝึกหัดที่ 1

> **หลักความซื่อสัตย์ทางวิชาการ (honour code):** ป้ายกำกับของภาพชุด val รวมอยู่ในชุดข้อมูลที่ดาวน์โหลดมา ผู้เรียนต้องส่งเฉพาะผลที่โมเดลของตนทำนายเท่านั้น ห้ามนำภาพชุด val ไปใช้ฝึกโมเดล และห้ามแก้ไขกรอบด้วยตนเอง

In [ ]:
# 🏆 Baseline — ส่งออกผลการตรวจจับบนชุด val และส่งผลไปยังตารางอันดับ (ไม่บังคับ)
# ============================================================================
# 🎯 วัตถุประสงค์ของเซลล์นี้ (โจทย์ท้าทาย ไม่บังคับ)
#   ส่งออกกรอบที่โมเดลทำนายบนภาพ MRI ชุดตรวจสอบ (val) ทั้ง 223 ภาพ เป็นไฟล์ JSON ชื่อ lab05_submission.json
#   แล้วส่งไฟล์นั้นไปยังตารางอันดับ (leaderboard) หากผู้เรียนกรอก NAME และ JOIN_CODE ในเซลล์การตั้งค่าแล้ว
#   เซิร์ฟเวอร์จะคำนวณคะแนน mAP@0.5 (หลักการเดียวกับ metrics.box.map50)
#   ใช้ความเชื่อมั่นขั้นต่ำต่ำมาก (conf=0.001) เพราะการคำนวณ mAP ต้องใช้กรอบทั้งหมดที่เรียงตามความเชื่อมั่น
#   ผลลัพธ์ : ข้อความบอกจำนวนกรอบ/จำนวนภาพที่ส่งออก และคะแนนจากตารางอันดับ (หรือข้อความแนะนำหากยังไม่กรอกข้อมูล)
# ============================================================================

# นำเข้าไลบรารีมาตรฐาน 2 ตัว
#   - json : อ่าน/เขียนข้อมูลในรูปแบบ JSON (รูปแบบข้อความมาตรฐานสำหรับแลกเปลี่ยนข้อมูล)
#   - os   : จัดการชื่อไฟล์และโฟลเดอร์
import json, os
# นำเข้าฟังก์ชัน check_det_dataset ของ ultralytics : อ่านไฟล์ตั้งค่าชุดข้อมูลและคืนตำแหน่งโฟลเดอร์ภาพ
from ultralytics.data.utils import check_det_dataset

# ----------------------------------------------------------------------------
# สร้างฟังก์ชัน export_detections — ให้โมเดลทำนายภาพชุด val แล้วบันทึกทุกกรอบเป็นไฟล์ JSON
#   - model : โมเดล YOLO ที่ต้องการใช้ทำนาย
#   - imgsz=416 : ขนาดภาพที่ป้อนให้โมเดล (ควรเท่ากับขนาดที่ใช้ฝึก)
#   - path="lab05_submission.json" : ชื่อไฟล์ผลลัพธ์ที่จะบันทึก
#   - **predict_kwargs : รับพารามิเตอร์แบบระบุชื่อเพิ่มเติมกี่ตัวก็ได้ (เก็บเป็น dict)
#                        แล้วส่งต่อให้ model.predict เช่น augment=True
#   - คืนค่า : ชื่อไฟล์ที่บันทึก
# ----------------------------------------------------------------------------
def export_detections(model, imgsz=416, path="lab05_submission.json", **predict_kwargs):
    # หาตำแหน่งโฟลเดอร์ภาพชุดตรวจสอบจากไฟล์ตั้งค่า brain-tumor.yaml (ค่าที่กุญแจ "val")
    val_dir = check_det_dataset("brain-tumor.yaml")["val"]
    # สร้างลิสต์ว่างไว้เก็บกรอบทั้งหมด
    dets = []
    # วนซ้ำผลการทำนายทีละภาพ
    #   - source=val_dir : โฟลเดอร์ภาพชุดตรวจสอบ
    #   - conf=0.001 : เก็บกรอบที่ความเชื่อมั่นตั้งแต่ 0.001 ขึ้นไป (แทบทุกกรอบ) เพื่อให้คำนวณ mAP ได้ถูกต้อง
    #   - imgsz=imgsz : ขนาดภาพตามพารามิเตอร์ของฟังก์ชัน
    #   - stream=True : ส่งผลออกมาทีละภาพ ประหยัดหน่วยความจำ
    #   - verbose=False : ไม่พิมพ์รายละเอียดของแต่ละภาพ
    #   - **predict_kwargs : กระจายพารามิเตอร์เพิ่มเติมที่ผู้เรียกส่งมา (ถ้ามี) ส่งต่อให้ predict
    for r in model.predict(source=val_dir, conf=0.001, imgsz=imgsz, stream=True, verbose=False, **predict_kwargs):
        # วนซ้ำทีละกรอบในภาพนี้ โดยจับคู่ (zip) พิกัด คลาส และความเชื่อมั่นของกรอบเดียวกัน
        #   - r.boxes.xyxy.tolist() : พิกัด (x1, y1, x2, y2) หน่วยพิกเซล แยกออกเป็น 4 ตัวแปรทันที
        #   - r.boxes.cls.tolist()  : หมายเลขคลาส -> c
        #   - r.boxes.conf.tolist() : ความเชื่อมั่น -> p
        for (x1, y1, x2, y2), c, p in zip(r.boxes.xyxy.tolist(), r.boxes.cls.tolist(), r.boxes.conf.tolist()):
            # เพิ่มกรอบนี้เป็น dict 1 รายการต่อท้ายลิสต์ dets ตามรูปแบบที่ตารางอันดับกำหนด
            #   - "image" : ชื่อไฟล์ภาพ (os.path.basename ตัดชื่อโฟลเดอร์ออก เหลือเฉพาะชื่อไฟล์)
            #   - "class" : หมายเลขคลาสเป็นจำนวนเต็ม
            #   - "conf"  : ความเชื่อมั่น ปัดเป็นทศนิยม 4 ตำแหน่งด้วย round(p, 4)
            #   - "bbox"  : พิกัดกรอบ [x1, y1, x2, y2] ปัดเป็นทศนิยม 1 ตำแหน่ง
            dets.append({"image": os.path.basename(r.path), "class": int(c), "conf": round(p, 4),
                         "bbox": [round(x1, 1), round(y1, 1), round(x2, 1), round(y2, 1)]})
    # พิมพ์สรุป: จำนวนกรอบ, จำนวนภาพที่ไม่ซ้ำ และชื่อไฟล์ผลลัพธ์
    #   - len(dets) : จำนวนกรอบทั้งหมด
    #   - {d["image"] for d in dets} : set comprehension รวบรวมชื่อภาพแบบไม่ซ้ำ แล้วนับด้วย len(...)
    print(len(dets), "กรอบ บน", len({d["image"] for d in dets}), "ภาพ ->", path)
    # เปิดไฟล์เพื่อเขียน ด้วยคำสั่ง with (ปิดไฟล์ให้อัตโนมัติเมื่อจบบล็อกที่ย่อหน้า แม้เกิดข้อผิดพลาด)
    #   - open(path, "w") : เปิดไฟล์ชื่อ path ในโหมด "w" (write = เขียนทับ/สร้างใหม่)
    #   - as f : ตั้งชื่อตัวแปรแทนไฟล์ที่เปิดว่า f
    with open(path, "w") as f:
        # เขียนลิสต์ dets ทั้งหมดลงไฟล์ f ในรูปแบบ JSON ด้วย json.dump(ข้อมูล, ไฟล์)
        json.dump(dets, f)
    # ส่งชื่อไฟล์ผลลัพธ์กลับไปให้ผู้เรียกใช้ฟังก์ชัน
    return path

# ----------------------------------------------------------------------------
# เรียกใช้ฟังก์ชันกับโมเดลที่ฝึกในแบบฝึกหัดที่ 1 แล้วส่งผลไปยังตารางอันดับ (ถ้ากรอกข้อมูลแล้ว)
# ----------------------------------------------------------------------------
# ส่งออกผลการตรวจจับของโมเดล det เป็นไฟล์ JSON แล้วเก็บชื่อไฟล์ไว้ในตัวแปร sub
#   - det : โมเดลที่ฝึกแล้ว
#   - imgsz=416 : ขนาดภาพเท่ากับตอนฝึก
sub = export_detections(det, imgsz=416)   # imgsz ต้องเท่ากับขนาดที่ใช้ฝึก
# ถ้ากรอกทั้ง NAME และ JOIN_CODE ในเซลล์การตั้งค่าแล้ว (ข้อความที่ไม่ว่างถือเป็นจริง)
if NAME and JOIN_CODE:
    # ส่งไฟล์ผลลัพธ์ไปยังตารางอันดับด้วยฟังก์ชัน submit_to_leaderboard (สร้างไว้ในเซลล์การตั้งค่า) และแสดงคะแนน
    submit_to_leaderboard(sub)
# มิฉะนั้น (ยังไม่ได้กรอกข้อมูล)
else:
    # พิมพ์ข้อความแนะนำวิธีกรอกข้อมูลเพื่อส่งผล
    print("โปรดระบุ NAME และ JOIN_CODE ในเซลล์การตั้งค่า แล้วรันเซลล์นั้นอีกครั้ง เพื่อส่งผลไปยังตารางอันดับ")

**แนวทางการปรับปรุงผลให้สูงกว่า baseline**
- ฝึกนานขึ้น (`epochs=50–100` โดยคำนึงถึงข้อจำกัดด้านเวลาของ Colab รุ่นฟรี) และ/หรือใช้ความละเอียดดั้งเดิมของภาพ `imgsz=512`
- ใช้โมเดลขนาดใหญ่ขึ้น เช่น `yolo11s.pt` หรือ `yolo11m.pt` (หรือตระกูลที่ใหม่กว่า เช่น `yolo26n.pt` หาก `ultralytics` เวอร์ชันที่ใช้รองรับ)
- ปรับการเสริมข้อมูล (data augmentation) เช่น `degrees`, `scale`, `flipud`, `mixup` และอัตราการเรียนรู้ (`lr0`, `cos_lr=True`)
- ใช้การเสริมข้อมูลขณะทดสอบ (test-time augmentation) ในขั้นการทำนาย: `export_detections(model, imgsz=..., augment=True)`
- ใช้ค่า `conf` ต่ำมากในไฟล์ที่ส่งเสมอ เนื่องจาก threshold ที่สูงขึ้นทำให้ค่า mAP ลดลงเท่านั้น

In [ ]:
# 🏆 โมเดลที่ปรับปรุงแล้ว (ไม่บังคับ) — กำหนด IMPROVE = True และปรับการตั้งค่าตามต้องการ
# ============================================================================
# 🎯 วัตถุประสงค์ของเซลล์นี้ (โจทย์ท้าทาย ไม่บังคับ)
#   ตัวอย่างการปรับปรุงโมเดลให้ได้คะแนนสูงกว่า baseline โดย
#   - ใช้โมเดลขนาดใหญ่ขึ้น (yolo11s = small) แทน nano
#   - ฝึกนานขึ้น (50 epoch) และใช้ภาพความละเอียดสูงขึ้น (512 พิกเซล)
#   จากนั้นประเมินผล (mAP50) ส่งออกไฟล์ผลลัพธ์ และส่งไปยังตารางอันดับ
#   ค่าเริ่มต้น IMPROVE = False จึงไม่มีการทำงานใด ๆ (ป้องกันการฝึกนานโดยไม่ตั้งใจ)
#   หากต้องการทดลอง ให้เปลี่ยนเป็น IMPROVE = True (อาจใช้เวลานานบน Colab รุ่นฟรี)
# ============================================================================

# สวิตช์เปิด/ปิดการทำงานของเซลล์: False = ไม่ทำงาน, True = ฝึกโมเดลที่ปรับปรุงแล้ว
#   (True และ False คือค่าความจริงทางตรรกะ (boolean) ใน Python)
IMPROVE = False

# ทำคำสั่งที่ย่อหน้าด้านล่างเฉพาะเมื่อ IMPROVE เป็น True
if IMPROVE:
    # สร้างโมเดลจากไฟล์โมเดลขนาด small ที่ฝึกล่วงหน้า (yolo11s.pt) ใหญ่และแม่นยำกว่า nano แต่ช้ากว่า
    det2 = YOLO("yolo11s.pt")                       # ทดลองโมเดลขนาดใหญ่ขึ้น
    # ฝึกโมเดลด้วยภาพ MRI เนื้องอกสมอง
    #   - data="brain-tumor.yaml" : ชุดข้อมูลเดียวกับก่อนหน้า
    #   - epochs=50 : ฝึก 50 รอบ (นานกว่า baseline)
    #   - imgsz=512 : ใช้ภาพขนาด 512 x 512 พิกเซล (ความละเอียดดั้งเดิมของภาพ)
    #   - batch=16 : ประมวลผลครั้งละ 16 ภาพ
    det2.train(data="brain-tumor.yaml", epochs=50, imgsz=512, batch=16)
    # ประเมินผลบนภาพชุดตรวจสอบแล้วพิมพ์ค่า mAP50
    #   - det2.val(data=..., imgsz=512) : ประเมินผลด้วยขนาดภาพเท่ากับตอนฝึก
    #   - .box.map50 : ดึงค่า mAP50 จากผลการประเมินต่อทันที (เรียกต่อกันเป็นลูกโซ่)
    print("mAP50 บนชุด val:", det2.val(data="brain-tumor.yaml", imgsz=512).box.map50)
    # ส่งออกผลการตรวจจับของโมเดลใหม่เป็นไฟล์ JSON (ฟังก์ชันจากเซลล์ Baseline) ด้วย imgsz=512 ให้ตรงกับตอนฝึก
    sub = export_detections(det2, imgsz=512)
    # ถ้ากรอก NAME และ JOIN_CODE แล้ว
    if NAME and JOIN_CODE:
        # ส่งไฟล์ผลลัพธ์ไปยังตารางอันดับและแสดงคะแนน
        submit_to_leaderboard(sub)

## ชิ้นงานที่ต้องส่ง (งานเดี่ยว)

ให้ส่งตามช่องทางที่รายวิชากำหนด ดังนี้
1. โน้ตบุ๊กนี้ที่ **รันครบทุกเซลล์ตามลำดับจากบนลงล่าง** (แสดงกราฟการฝึก ค่า mAP50 / mAP50-95 และภาพ MRI ที่มีกรอบการตรวจจับอย่างน้อยหนึ่งภาพ)
2. คำตอบของ ✍️ **คำถามที่ 1–4**

🏆 *ไม่บังคับ:* ผู้เรียนที่ทำโจทย์ท้าทายบนตารางอันดับ สามารถระบุชื่อที่ใช้บนตารางอันดับและคะแนนสูงสุดไว้ในส่วนนี้ได้

✍️ **ชื่อบนตารางอันดับและคะแนน (ไม่บังคับ):** 